In [ ]:
# prompt: import drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import random


In [ ]:
!pip install noisereduce

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/SpeechFolder'  # Change this to your path
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Emotion code to label
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

In [ ]:
def generate_labels_csv(dataset_path):
    data = []
    for subfolder in os.listdir(dataset_path):
        subfolder_path = os.path.join(dataset_path, subfolder)
        # Skip non-directories
        if not os.path.isdir(subfolder_path):
            continue
        # Get the emotion label based on the subfolder name
        emotion_label = emotion_map.get(subfolder, None)
        if emotion_label:
            for file in os.listdir(subfolder_path):
                if file.endswith('.wav'):
                    data.append({'filename': os.path.join(subfolder, file), 'emotion': emotion_label})

    df = pd.DataFrame(data)
    df.to_csv(os.path.join(dataset_path, 'ravdess_labels.csv'), index=False)
    print("✅ Labels CSV generated!")


In [ ]:
def preprocess_audio(signal, sample_rate):
    # Normalize amplitude
    signal = librosa.util.normalize(signal)

    # Denoise (using Noisereduce)
    signal = nr.reduce_noise(y=signal, sr=sample_rate)

    # Augmentation: Random pitch shift
    if random.random() > 0.5:
        signal = librosa.effects.pitch_shift(signal, sample_rate, n_steps=random.randint(-2, 2))

    # Augmentation: Time stretching
    if random.random() > 0.5:
        signal = librosa.effects.time_stretch(signal, random.uniform(0.8, 1.2))

    return signal

In [ ]:
def extract_features(file_path):
    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    if len(signal) > MAX_LEN:
        signal = signal[:MAX_LEN]
    else:
        signal = np.pad(signal, (0, MAX_LEN - len(signal)))

    # Apply preprocessing steps to the signal
    signal = preprocess_audio(signal, sr)

    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    mfccs = np.mean(mfccs.T, axis=0)

    zcr = librosa.feature.zero_crossing_rate(signal)
    zcr = np.mean(zcr)

    rmse = librosa.feature.rms(y=signal)
    rmse = np.mean(rmse)

    return np.hstack([mfccs, zcr, rmse])


In [ ]:
def load_dataset(dataset_path):
    features, labels = [], []

    for file in os.listdir(dataset_path):
        if file.endswith('.wav'):
            parts = file.split('-')
            if len(parts) >= 3:
                gender_code = parts[2]
                gender = gender_map.get(gender_code)
                if gender:
                    file_path = os.path.join(dataset_path, file)
                    try:
                        feat = extract_features(file_path)
                        features.append(feat)
                        labels.append(gender)
                    except Exception as e:
                        print(f"⚠️ Failed to process {file}: {e}")

    return np.array(features), np.array(labels)

In [ ]:
def build_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(64))
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [ ]:
def main():
    print("🔍 Loading and processing data...")
    X, y = load_dataset(DATASET_PATH)

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    y_categorical = to_categorical(y_encoded)

    X = X[..., np.newaxis]  # Add channel dimension
    X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

    print("🧠 Training model...")
    model = build_model(input_shape=(X.shape[1], 1), num_classes=y_categorical.shape[1])
    model.summary()
    model.fit(X_train, y_train, validation_data=(X_test, y_test), batch_size=32, epochs=50)

    print("✅ Training complete!")

    # Save model
    model.save("emotion_recognition_model.h5")
    print("📁 Model saved as emotion_recognition_model.h5")

    # Test prediction
    test_path = os.path.join(DATASET_PATH, df['filename'].iloc[0])  # Use the first file for testing
    test_feat = extract_features(test_path).reshape(1, -1, 1)
    pred = model.predict(test_feat)
    pred_label = le.inverse_transform([np.argmax(pred)])
    print(f"🎤 Predicted Emotion: {pred_label[0]}")

if __name__ == "__main__":
    main()


🔍 Loading and processing data...


NameError: name 'gender_map' is not defined

In [ ]:
# prompt: import drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install noisereduce

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import random
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# -----------------------------
# Configuration
# -----------------------------
DATASET_PATH = '/content/drive/MyDrive/SpeechFolder'  # Change this to your path
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Emotion code to label
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

# -----------------------------
# Install necessary libraries
# -----------------------------
!pip install noisereduce

# -----------------------------
# Extract emotion labels from filenames
# -----------------------------

def generate_labels_csv(dataset_path):
    data = []
    # The original code iterated through subfolders, but the file structure in the second block
    # of code suggests files are directly in DATASET_PATH. Adopting the second block's approach.
    for file in os.listdir(dataset_path):
        if file.endswith('.wav'):
            parts = file.split('-')
            if len(parts) >= 3:
                emotion_code = parts[2]
                emotion = emotion_map.get(emotion_code)
                if emotion:
                    # In the second block, 'filename' just stored the file name, not the full path
                    data.append({'filename': file, 'emotion': emotion})
    df = pd.DataFrame(data)
    csv_path = os.path.join(dataset_path, 'ravdess_labels.csv')
    df.to_csv(csv_path, index=False)
    print(f"✅ Labels CSV generated at {csv_path}!")

# -----------------------------
# Preprocessing - Normalization, Denoising, Augmentation
# -----------------------------

def preprocess_audio(signal, sample_rate):
    # Normalize amplitude
    signal = librosa.util.normalize(signal)

    # Denoise (using Noisereduce)
    signal = nr.reduce_noise(y=signal, sr=sample_rate)

    # Augmentation: Random pitch shift
    if random.random() > 0.5:
        # Ensure sample_rate is an integer for pitch_shift
        signal = librosa.effects.pitch_shift(y=signal, sr=int(sample_rate), n_steps=random.randint(-2, 2))

    # Augmentation: Time stretching
    if random.random() > 0.5:
        # Ensure rate is within valid range and time_stretch expects float
        signal = librosa.effects.time_stretch(y=signal, rate=random.uniform(0.8, 1.2))

    return signal

# -----------------------------
# Feature extraction: MFCC + ZCR + RMSE
# -----------------------------

def extract_features(file_path):
    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    if len(signal) > MAX_LEN:
        signal = signal[:MAX_LEN]
    else:
        signal = np.pad(signal, (0, MAX_LEN - len(signal)))

    # Apply preprocessing steps to the signal
    # Pass the correct sample rate used during loading
    signal = preprocess_audio(signal, sr)

    # Ensure sr is passed to librosa feature extraction functions
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    mfccs = np.mean(mfccs.T, axis=0)

    zcr = librosa.feature.zero_crossing_rate(y=signal) # zcr also needs y argument
    zcr = np.mean(zcr)

    rmse = librosa.feature.rms(y=signal)
    rmse = np.mean(rmse)

    return np.hstack([mfccs, zcr, rmse])

# -----------------------------
# Load data and extract features
# -----------------------------

# Modified load_dataset to read from the generated CSV
def load_dataset(dataset_path, df): # Added df as an argument
    features, labels = [], []

    for _, row in df.iterrows():
        file_path = os.path.join(dataset_path, row['filename'])
        if os.path.exists(file_path):
            try:
                feat = extract_features(file_path)
                features.append(feat)
                labels.append(row['emotion'])
            except Exception as e:
                print(f"⚠️ Failed to process {file_path}: {e}")

    return np.array(features), np.array(labels)

# -----------------------------
# Build the CNN + LSTM model
# -----------------------------

def build_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(64))
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# -----------------------------
# Main pipeline
# -----------------------------

def main():
    # Generate the labels CSV first if it doesn't exist
    csv_path = os.path.join(DATASET_PATH, 'ravdess_labels.csv')
    if not os.path.exists(csv_path):
        print("Generating labels CSV...")
        generate_labels_csv(DATASET_PATH)

    # Load the DataFrame into the main scope
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: Labels CSV not found at {csv_path}. Please ensure the DATASET_PATH is correct and contains audio files.")
        return
    except pd.errors.EmptyDataError:
        print(f"Error: Labels CSV at {csv_path} is empty. Please ensure your DATASET_PATH contains valid audio files with correct naming convention.")
        return


    print("🔍 Loading and processing data...")
    # Pass the loaded DataFrame to load_dataset
    X, y = load_dataset(DATASET_PATH, df)

    # Check if any data was loaded
    if len(X) == 0:
        print("❌ No audio files processed. Please check DATASET_PATH and file naming.")
        return

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    y_categorical = to_categorical(y_encoded)

    X = X[..., np.newaxis]  # Add channel dimension
    X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded) # Added stratify

    print("🧠 Training model...")
    model = build_model(input_shape=(X.shape[1], 1), num_classes=y_categorical.shape[1])
    model.summary()
    model.fit(X_train, y_train, validation_data=(X_test, y_test), batch_size=32, epochs=50)

    print("✅ Training complete!")

    # Save model
    model.save("emotion_recognition_model.h5")
    print("📁 Model saved as emotion_recognition_model.h5")

    # Test prediction
    # Ensure df is not empty before trying to access its elements
    if not df.empty:
        test_path = os.path.join(DATASET_PATH, df['filename'].iloc[0])  # Use the first file for testing
        test_feat = extract_features(test_path).reshape(1, -1, 1)
        pred = model.predict(test_feat)
        pred_label = le.inverse_transform([np.argmax(pred)])
        print(f"🎤 Predicted Emotion: {pred_label[0]}")
    else:
        print("⚠️ Cannot perform test prediction: DataFrame is empty.")


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Loading and processing data...
🧠 Training model...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 13, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 4, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,048 (308.78 KB)

 Trainable params: 79,048 (308.78 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.1099 - loss: 2.0970 - val_accuracy: 0.1635 - val_loss: 2.0451
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.1860 - loss: 2.0418 - val_accuracy: 0.1731 - val_loss: 2.0305
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1794 - loss: 2.0215 - val_accuracy: 0.1490 - val_loss: 2.0224
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2383 - loss: 1.9654 - val_accuracy: 0.2788 - val_loss: 1.9704
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2360 - loss: 1.9277 - val_accuracy: 0.2356 - val_loss: 1.9512
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2269 - loss: 1.9189 - val_accuracy: 0.2260 - val_loss: 1.9303
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2700 - loss: 1.8414 - val_accuracy: 0.2260 - val_loss: 1.8975
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2699 - loss: 1.8418 - val_accuracy: 0.2548 - v

✅ Training complete!
📁 Model saved as emotion_recognition_model.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step
🎤 Predicted Emotion: happy


In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.5 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import librosa
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import noisereduce as nr
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
import io

# -------------------------
# Load model and encoder
# -------------------------
model = load_model("emotion_recognition_model.h5")
emotion_labels = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
le = LabelEncoder()
le.fit(emotion_labels)

SAMPLE_RATE = 22050
MAX_LEN = SAMPLE_RATE * 3

# -------------------------
# Audio Processing Functions
# -------------------------
def preprocess_audio(signal, sr):
    signal = librosa.util.normalize(signal)
    signal = nr.reduce_noise(y=signal, sr=sr)

    # Optional augmentations
    if np.random.rand() > 0.5:
        signal = librosa.effects.pitch_shift(signal, sr=sr, n_steps=np.random.randint(-2, 3))
    if np.random.rand() > 0.5:
        signal = librosa.effects.time_stretch(signal, rate=np.random.uniform(0.8, 1.2))

    return signal

def extract_features(signal, sr):
    if len(signal) > MAX_LEN:
        signal = signal[:MAX_LEN]
    else:
        signal = np.pad(signal, (0, MAX_LEN - len(signal)))

    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs.T, axis=0)

    zcr = librosa.feature.zero_crossing_rate(y=signal)
    zcr_mean = np.mean(zcr)

    rmse = librosa.feature.rms(y=signal)
    rmse_mean = np.mean(rmse)

    return np.hstack([mfccs_mean, zcr_mean, rmse_mean])

# -------------------------
# Plotting functions
# -------------------------
def plot_mel_spectrogram(signal, sr):
    mel = librosa.feature.melspectrogram(y=signal, sr=sr)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, ax = plt.subplots(figsize=(8, 3))
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=ax)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    ax.set(title='Mel Spectrogram')
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    return buf

def plot_mfcc(signal, sr):
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    fig, ax = plt.subplots(figsize=(8, 3))
    img = librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax)
    ax.set(title='MFCC')
    fig.colorbar(img, ax=ax)
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    return buf

def plot_prediction_pie(prediction_probs):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.pie(prediction_probs, labels=emotion_labels, autopct='%1.1f%%', startangle=140)
    ax.set_title("Emotion Prediction Probabilities")
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    return buf

# -------------------------
# Main Prediction Function
# -------------------------
def predict_emotion(audio):
    # Load audio from Gradio
    signal, sr = librosa.load(audio, sr=SAMPLE_RATE)

    # Save copies for visualization
    signal_original = signal.copy()
    mel_img = plot_mel_spectrogram(signal_original, sr)
    mfcc_img = plot_mfcc(signal_original, sr)

    # Preprocess and extract features
    signal_processed = preprocess_audio(signal.copy(), sr)
    features = extract_features(signal_processed, sr)
    features = features.reshape(1, -1, 1)

    # Predict
    prediction = model.predict(features)[0]
    predicted_label = le.inverse_transform([np.argmax(prediction)])[0]
    pie_img = plot_prediction_pie(prediction)

    return audio, mel_img, mfcc_img, pie_img, predicted_label

# -------------------------
# Gradio Interface
# -------------------------
interface = gr.Interface(
    fn=predict_emotion,
    # Changed 'source' to 'sources' and set it to a list ["microphone", "upload"]
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="🎤 Record or Upload Speech"),
    outputs=[
        gr.Audio(label="🔊 Original Audio"),
        gr.Image(label="🎼 Mel Spectrogram"),
        gr.Image(label="🎙️ MFCCs"),
        gr.Image(label="📊 Prediction Pie"),
        gr.Label(label="🎯 Final Predicted Emotion")
    ],
    title="🎵 Emotion Recognition from Speech",
    description="Record or upload a short 3-second audio and get emotion prediction with visualizations."
)

interface.launch()

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'emotion_recognition_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import random
import matplotlib.pyplot as plt
# Remove Google Drive imports if not needed locally
# from google.colab import drive

# -----------------------------
# Configuration
# -----------------------------
# Update this path to your local folder containing audio files
DATASET_PATH = '/content/drive/MyDrive/SpeechFolder'  # Change this to your path
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Emotion code to label
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

# Install necessary packages if needed
# import subprocess
# subprocess.call(['pip', 'install', 'noisereduce'])

# -----------------------------
# Extract emotion labels from filenames
# -----------------------------
def generate_labels_csv(dataset_path):
    data = []
    files_found = 0

    for file in os.listdir(dataset_path):
        if file.endswith('.wav'):
            files_found += 1
            parts = file.split('-')
            if len(parts) >= 3:
                emotion_code = parts[2]
                emotion = emotion_map.get(emotion_code)
                if emotion:
                    data.append({'filename': file, 'emotion': emotion})

    if files_found == 0:
        print(f"⚠️ No WAV files found in {dataset_path}!")
        print(f"Current directory: {os.getcwd()}")
        print(f"Files in directory: {os.listdir('.')}")
        return None

    df = pd.DataFrame(data)
    csv_path = os.path.join(dataset_path, 'ravdess_labels.csv')
    df.to_csv(csv_path, index=False)

    # Print distribution of emotions
    emotion_counts = df['emotion'].value_counts()
    print("\n--- Emotion Distribution ---")
    for emotion, count in emotion_counts.items():
        print(f"{emotion}: {count} files")

    print(f"\n✅ Labels CSV generated at {csv_path} with {len(df)} files!")
    return df

# -----------------------------
# Preprocessing - Normalization, Denoising, Augmentation
# -----------------------------
def preprocess_audio(signal, sample_rate):
    # Normalize amplitude
    signal = librosa.util.normalize(signal)

    # Denoise (using Noisereduce)
    signal = nr.reduce_noise(y=signal, sr=sample_rate)

    # Augmentation: Random pitch shift
    if random.random() > 0.5:
        signal = librosa.effects.pitch_shift(
            y=signal, sr=int(sample_rate), n_steps=random.randint(-2, 2)
        )

    # Augmentation: Time stretching
    if random.random() > 0.5:
        signal = librosa.effects.time_stretch(
            y=signal, rate=random.uniform(0.8, 1.2)
        )

    return signal

# -----------------------------
# Feature extraction: MFCC + ZCR + RMSE
# -----------------------------
def extract_features(file_path):
    try:
        signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)

        if len(signal) > MAX_LEN:
            signal = signal[:MAX_LEN]
        else:
            signal = np.pad(signal, (0, MAX_LEN - len(signal)))

        # Apply preprocessing steps to the signal
        signal = preprocess_audio(signal, sr)

        # Extract features
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
        mfccs = np.mean(mfccs.T, axis=0)

        zcr = librosa.feature.zero_crossing_rate(y=signal)
        zcr = np.mean(zcr)

        rmse = librosa.feature.rms(y=signal)
        rmse = np.mean(rmse)

        return np.hstack([mfccs, zcr, rmse])

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# -----------------------------
# Load data and extract features
# -----------------------------
def load_dataset(dataset_path, df):
    features, labels = [], []
    total_files = len(df)

    print(f"\nProcessing {total_files} audio files...")

    for i, (_, row) in enumerate(df.iterrows()):
        if i % 10 == 0:  # Progress update every 10 files
            print(f"Processing file {i+1}/{total_files}...")

        file_path = os.path.join(dataset_path, row['filename'])
        if os.path.exists(file_path):
            try:
                feat = extract_features(file_path)
                if feat is not None:
                    features.append(feat)
                    labels.append(row['emotion'])
            except Exception as e:
                print(f"⚠️ Failed to process {file_path}: {e}")
        else:
            print(f"⚠️ File not found: {file_path}")

    if len(features) == 0:
        print("❌ No features extracted! Check file paths and audio format.")
        return None, None

    print(f"✅ Successfully processed {len(features)}/{total_files} files")
    return np.array(features), np.array(labels)

# -----------------------------
# Build the CNN + LSTM model
# -----------------------------
def build_model(input_shape, num_classes):
    model = Sequential([
        Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Conv1D(128, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

# -----------------------------
# Plotting functions for visualization
# -----------------------------
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # Plot accuracy
    ax1.plot(history.history['accuracy'], label='Training')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title('Model Accuracy')
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend(loc='lower right')
    ax1.grid(True)

    # Plot loss
    ax2.plot(history.history['loss'], label='Training')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title('Model Loss')
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend(loc='upper right')
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()
    print("✅ Training history plot saved as 'training_history.png'")

def plot_confusion_matrix(y_true, y_pred, classes):
    from sklearn.metrics import confusion_matrix
    import seaborn as sns

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title('Normalized Confusion Matrix')
    plt.ylabel('True Emotion')
    plt.xlabel('Predicted Emotion')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()
    print("✅ Confusion matrix plot saved as 'confusion_matrix.png'")

# -----------------------------
# Main pipeline
# -----------------------------
def main():
    print("🎵 Speech Emotion Recognition - Model Training 🎵")
    print("------------------------------------------------")

    # Check if DATASET_PATH exists
    if not os.path.exists(DATASET_PATH):
        print(f"❌ Error: Dataset path '{DATASET_PATH}' does not exist!")
        print(f"Current directory: {os.getcwd()}")
        print(f"Files in directory: {os.listdir('.')}")
        return

    # Generate the labels CSV if it doesn't exist
    csv_path = os.path.join(DATASET_PATH, 'ravdess_labels.csv')
    if os.path.exists(csv_path):
        print(f"Loading existing labels CSV from {csv_path}...")
        df = pd.read_csv(csv_path)
    else:
        print("Generating labels CSV...")
        df = generate_labels_csv(DATASET_PATH)
        if df is None:
            return

    print("\n🔍 Loading and processing data...")
    X, y = load_dataset(DATASET_PATH, df)

    if X is None or len(X) == 0:
        print("❌ No audio files processed. Please check DATASET_PATH and file naming.")
        return

    print(f"\n📊 Feature shape: {X.shape}")
    print(f"📊 Number of classes: {len(np.unique(y))}")

    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    y_categorical = to_categorical(y_encoded)

    # Add channel dimension for CNN
    X = X[..., np.newaxis]

    # Split data with stratification to maintain class distribution
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
    )

    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Testing set: {X_test.shape[0]} samples")

    # Define callbacks for training
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=10, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5, min_lr=0.0001
        )
    ]

    print("\n🧠 Training model...")
    model = build_model(input_shape=(X.shape[1], 1), num_classes=y_categorical.shape[1])
    model.summary()

    # Start training
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        batch_size=32,
        epochs=50,
        callbacks=callbacks,
        verbose=1
    )

    print("\n✅ Training complete!")

    # Evaluate model
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test accuracy: {accuracy:.4f}")
    print(f"Test loss: {loss:.4f}")

    # Plot training history
    plot_training_history(history)

    # Generate confusion matrix
    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)
    plot_confusion_matrix(y_true, y_pred, le.classes_)

    # Save model
    model.save("emotion_recognition_model.h5")
    print("📁 Model saved as emotion_recognition_model.h5")

    # Test prediction on a sample
    if len(df) > 0:
        test_path = os.path.join(DATASET_PATH, df['filename'].iloc[0])
        if os.path.exists(test_path):
            test_feat = extract_features(test_path).reshape(1, -1, 1)
            pred = model.predict(test_feat)
            pred_label = le.inverse_transform([np.argmax(pred)])
            print(f"🎤 Sample test prediction: {pred_label[0]}")
        else:
            print(f"⚠️ Test file not found: {test_path}")

if __name__ == "__main__":
    main()

🎵 Speech Emotion Recognition - Model Training 🎵
------------------------------------------------
Loading existing labels CSV from /content/drive/MyDrive/SpeechFolder/ravdess_labels.csv...

🔍 Loading and processing data...

Processing 1036 audio files...
Processing file 1/1036...
Processing file 11/1036...
Processing file 21/1036...
Processing file 31/1036...
Processing file 41/1036...
Processing file 51/1036...
Processing file 61/1036...
Processing file 71/1036...
Processing file 81/1036...
Processing file 91/1036...
Processing file 101/1036...
Processing file 111/1036...
Processing file 121/1036...
Processing file 131/1036...
Processing file 141/1036...
Processing file 151/1036...
Processing file 161/1036...
Processing file 171/1036...
Processing file 181/1036...
Processing file 191/1036...
Processing file 201/1036...
Processing file 211/1036...
Processing file 221/1036...
Processing file 231/1036...
Processing file 241/1036...
Processing file 251/1036...
Processing file 261/1036...
P

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 13, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 4, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,048 (308.78 KB)

 Trainable params: 79,048 (308.78 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.1376 - loss: 2.0980 - val_accuracy: 0.1779 - val_loss: 2.0527 - learning_rate: 0.0010
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1884 - loss: 2.0415 - val_accuracy: 0.2067 - val_loss: 2.0352 - learning_rate: 0.0010
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.2546 - loss: 1.9940 - val_accuracy: 0.2212 - val_loss: 2.0209 - learning_rate: 0.0010
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2083 - loss: 2.0000 - val_accuracy: 0.1923 - val_loss: 2.0075 - learning_rate: 0.0010
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.2304 - loss: 1.9564 - val_accuracy: 0.2308 - val_loss: 1.9338 - learning_rate: 0.0010
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2362 - loss: 1.8983 - val_accuracy: 0.2260 - val_loss: 1.9219 - learning_rate: 0.0010
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.2997 - loss: 1.8468 - val_acc

✅ Confusion matrix plot saved as 'confusion_matrix.png'
📁 Model saved as emotion_recognition_model.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
🎤 Sample test prediction: fearful


In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow as tf
import random
from google.colab import drive

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# Mount Google Drive
drive.mount('/content/drive')

# -----------------------------
# Configuration
# -----------------------------
DATASET_PATH = '/content/drive/MyDrive/SpeechFolder'  # Change this to your path
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Emotion code to label
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

# -----------------------------
# Install necessary libraries
# -----------------------------
!pip install noisereduce librosa sklearn tensorflow matplotlib

# -----------------------------
# Extract emotion labels from filenames
# -----------------------------

def generate_labels_csv(dataset_path):
    """Generate a CSV file containing filenames and their emotion labels."""
    data = []
    valid_files = 0
    skipped_files = 0

    for file in os.listdir(dataset_path):
        if file.endswith('.wav'):
            parts = file.split('-')
            if len(parts) >= 3:
                emotion_code = parts[2]
                emotion = emotion_map.get(emotion_code)
                if emotion:
                    data.append({'filename': file, 'emotion': emotion})
                    valid_files += 1
                else:
                    skipped_files += 1
            else:
                skipped_files += 1

    df = pd.DataFrame(data)
    csv_path = os.path.join(dataset_path, 'ravdess_labels.csv')
    df.to_csv(csv_path, index=False)

    print(f"✅ Labels CSV generated at {csv_path}!")
    print(f"📊 Files processed: {valid_files} valid, {skipped_files} skipped")

    # Print distribution of emotions
    emotion_counts = df['emotion'].value_counts()
    for emotion, count in emotion_counts.items():
        print(f"   {emotion}: {count} files")

    return df

# -----------------------------
# Preprocessing - Normalization, Denoising, Augmentation
# -----------------------------

def preprocess_audio(signal, sample_rate):
    """Apply audio preprocessing techniques to enhance signal quality."""
    # Normalize amplitude
    signal = librosa.util.normalize(signal)

    # Denoise (using Noisereduce)
    signal = nr.reduce_noise(y=signal, sr=sample_rate)

    return signal

def augment_audio(signal, sample_rate):
    """Apply data augmentation techniques to increase training data diversity."""
    # Original signal
    augmented_signals = [signal]

    # Pitch shift up
    pitch_up = librosa.effects.pitch_shift(y=signal, sr=int(sample_rate), n_steps=2)
    augmented_signals.append(pitch_up)

    # Pitch shift down
    pitch_down = librosa.effects.pitch_shift(y=signal, sr=int(sample_rate), n_steps=-2)
    augmented_signals.append(pitch_down)

    # Time stretching - faster
    stretch_fast = librosa.effects.time_stretch(y=signal, rate=1.2)
    # Ensure it fits MAX_LEN
    if len(stretch_fast) > MAX_LEN:
        stretch_fast = stretch_fast[:MAX_LEN]
    else:
        stretch_fast = np.pad(stretch_fast, (0, MAX_LEN - len(stretch_fast)))
    augmented_signals.append(stretch_fast)

    # Time stretching - slower
    stretch_slow = librosa.effects.time_stretch(y=signal, rate=0.8)
    # Ensure it fits MAX_LEN
    if len(stretch_slow) > MAX_LEN:
        stretch_slow = stretch_slow[:MAX_LEN]
    else:
        stretch_slow = np.pad(stretch_slow, (0, MAX_LEN - len(stretch_slow)))
    augmented_signals.append(stretch_slow)

    return augmented_signals

# -----------------------------
# Feature extraction: MFCC + ZCR + RMSE + Spectral Centroid + Chroma
# -----------------------------

def extract_features(signal, sr):
    """Extract audio features for emotion recognition."""
    if len(signal) > MAX_LEN:
        signal = signal[:MAX_LEN]
    else:
        signal = np.pad(signal, (0, MAX_LEN - len(signal)))

    # Extract MFCCs (increased to 20 for better feature representation)
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
    mfccs = np.mean(mfccs.T, axis=0)

    # Extract delta MFCCs (1st order derivatives)
    delta_mfccs = librosa.feature.delta(mfccs)

    # Zero-crossing rate - voice characteristics
    zcr = librosa.feature.zero_crossing_rate(y=signal)
    zcr = np.mean(zcr)

    # Root mean square energy - volume/intensity
    rmse = librosa.feature.rms(y=signal)
    rmse = np.mean(rmse)

    # Spectral centroid - brightness of sound
    spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr)
    spectral_centroid = np.mean(spectral_centroid)

    # Chroma features - tonal content
    chroma = librosa.feature.chroma_stft(y=signal, sr=sr)
    chroma = np.mean(chroma.T, axis=0)

    # Spectral contrast - difference between peaks and valleys
    spectral_contrast = librosa.feature.spectral_contrast(y=signal, sr=sr)
    spectral_contrast = np.mean(spectral_contrast.T, axis=0)

    # Combine all features
    return np.hstack([
        mfccs,            # 20 features
        delta_mfccs,      # 20 features
        zcr,              # 1 feature
        rmse,             # 1 feature
        spectral_centroid,# 1 feature
        chroma,           # 12 features
        spectral_contrast # 7 features
    ])

# -----------------------------
# Load data and extract features
# -----------------------------

def load_dataset(dataset_path, df, use_augmentation=True):
    """Load audio files, extract features, and apply augmentation if requested."""
    features, labels = [], []
    processed = 0
    total_files = len(df)

    for idx, row in df.iterrows():
        file_path = os.path.join(dataset_path, row['filename'])
        if os.path.exists(file_path):
            try:
                # Load and preprocess audio
                signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
                signal_processed = preprocess_audio(signal, sr)

                # Extract features from original processed signal
                feat = extract_features(signal_processed, sr)
                features.append(feat)
                labels.append(row['emotion'])

                # Apply augmentation if requested
                if use_augmentation:
                    augmented_signals = augment_audio(signal_processed, sr)
                    # Skip first signal as it's the original one
                    for aug_signal in augmented_signals[1:]:
                        aug_feat = extract_features(aug_signal, sr)
                        features.append(aug_feat)
                        labels.append(row['emotion'])

                processed += 1
                if processed % 10 == 0:
                    print(f"Processed {processed}/{total_files} files...")

            except Exception as e:
                print(f"⚠️ Failed to process {file_path}: {e}")

    print(f"✅ Processed {processed} files, created {len(features)} feature vectors")
    return np.array(features), np.array(labels)

# -----------------------------
# Build an improved model: CNN + LSTM with attention
# -----------------------------

def build_model(input_shape, num_classes):
    """Build a CNN-LSTM model architecture with attention mechanism."""
    model = Sequential()

    # First Conv Block
    model.add(Conv1D(64, kernel_size=5, strides=1, padding='same', activation='relu', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    # Second Conv Block
    model.add(Conv1D(128, kernel_size=3, strides=1, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    # Third Conv Block
    model.add(Conv1D(256, kernel_size=3, strides=1, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    # LSTM layer
    model.add(LSTM(128, return_sequences=False))
    model.add(Dropout(0.4))

    # Dense layers
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    # Compile with Adam optimizer and categorical crossentropy loss
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=['accuracy']
    )

    return model

# -----------------------------
# Main pipeline
# -----------------------------

def main():
    print("🚀 Starting Speech Emotion Recognition Model Training")

    # Generate the labels CSV if it doesn't exist
    csv_path = os.path.join(DATASET_PATH, 'ravdess_labels.csv')
    if os.path.exists(csv_path):
        print(f"📄 Loading existing labels from {csv_path}")
        df = pd.read_csv(csv_path)
    else:
        print("📝 Generating labels CSV...")
        df = generate_labels_csv(DATASET_PATH)

    # Check if DataFrame is empty
    if df.empty:
        print("❌ No valid audio files found. Please check your dataset path and file naming convention.")
        return

    print("🔍 Loading and processing data...")
    X, y = load_dataset(DATASET_PATH, df, use_augmentation=True)

    # Check if any data was loaded
    if len(X) == 0:
        print("❌ No audio files processed. Please check DATASET_PATH and file naming.")
        return

    print(f"✅ Dataset loaded: {X.shape[0]} samples")

    # Encode labels
    print("🏷️ Encoding labels...")
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    y_categorical = to_categorical(y_encoded)
    print(f"✅ Label mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")

    # Reshape features for CNN input: (samples, timesteps, features)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    # Split data into train and test sets
    print("🔪 Splitting data into train and test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
    )
    print(f"✅ Train set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")

    # Build and train the model
    print("🧠 Building model architecture...")
    model = build_model(input_shape=(X.shape[1], 1), num_classes=y_categorical.shape[1])
    model.summary()

    # Set up callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001),
        ModelCheckpoint('best_emotion_model.h5', monitor='val_accuracy', save_best_only=True, mode='max')
    ]

    # Train the model
    print("🏋️ Training model...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        batch_size=32,
        epochs=100,
        callbacks=callbacks,
        verbose=1
    )

    print("✅ Training complete!")

    # Evaluate the model
    test_loss, test_acc = model.evaluate(X_test, y_test)
    print(f"📊 Test accuracy: {test_acc:.4f}")
    print(f"📊 Test loss: {test_loss:.4f}")

    # Save model
    model.save("emotion_recognition_model.h5")
    print("📁 Model saved as emotion_recognition_model.h5")

    # Save label encoder classes
    np.save('emotion_classes.npy', le.classes_)
    print("📁 Label encoder classes saved as emotion_classes.npy")

    # Predict on a test sample
    if not df.empty:
        test_path = os.path.join(DATASET_PATH, df['filename'].iloc[0])
        signal, sr = librosa.load(test_path, sr=SAMPLE_RATE)
        signal_processed = preprocess_audio(signal, sr)
        test_feat = extract_features(signal_processed, sr).reshape(1, -1, 1)
        pred = model.predict(test_feat)
        pred_label = le.inverse_transform([np.argmax(pred)])
        print(f"🎤 Sample Prediction Test:")
        print(f"   File: {df['filename'].iloc[0]}")
        print(f"   True emotion: {df['emotion'].iloc[0]}")
        print(f"   Predicted emotion: {pred_label[0]}")
        print(f"   Confidence: {np.max(pred):.2f}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'noisereduce'

In [ ]:
import gradio as gr
import librosa
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import noisereduce as nr
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
import io
import os

# -------------------------
# Load model and encoder
# -------------------------
# Check if model exists, otherwise provide clear message
model_path = "emotion_recognition_model.h5"
try:
    model = load_model(model_path)
    model_loaded = True
except (IOError, ImportError):
    print(f"⚠️ Model not found at {model_path}. You'll need to train the model first.")
    model_loaded = False

emotion_labels = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
le = LabelEncoder()
le.fit(emotion_labels)

SAMPLE_RATE = 22050
MAX_LEN = SAMPLE_RATE * 3

# -------------------------
# Audio Processing Functions
# -------------------------
def preprocess_audio(signal, sr):
    signal = librosa.util.normalize(signal)
    signal = nr.reduce_noise(y=signal, sr=sr)
    return signal

def extract_features(signal, sr):
    if len(signal) > MAX_LEN:
        signal = signal[:MAX_LEN]
    else:
        signal = np.pad(signal, (0, MAX_LEN - len(signal)))

    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs.T, axis=0)

    zcr = librosa.feature.zero_crossing_rate(y=signal)
    zcr_mean = np.mean(zcr)

    rmse = librosa.feature.rms(y=signal)
    rmse_mean = np.mean(rmse)

    return np.hstack([mfccs_mean, zcr_mean, rmse_mean])

# -------------------------
# Plotting functions
# -------------------------
def plot_mel_spectrogram(signal, sr):
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(
        librosa.power_to_db(librosa.feature.melspectrogram(y=signal, sr=sr), ref=np.max),
        y_axis='mel', x_axis='time', sr=sr
    )
    plt.colorbar(format='%+2.0f dB')
    plt.title('Mel Spectrogram')
    plt.tight_layout()

    # Save to a temporary file instead of BytesIO for better compatibility
    temp_path = "temp_mel_spec.png"
    plt.savefig(temp_path)
    plt.close()
    return temp_path

def plot_mfcc(signal, sr):
    plt.figure(figsize=(10, 4))
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    librosa.display.specshow(mfccs, x_axis='time', sr=sr)
    plt.colorbar()
    plt.title('MFCC')
    plt.tight_layout()

    # Save to a temporary file
    temp_path = "temp_mfcc.png"
    plt.savefig(temp_path)
    plt.close()
    return temp_path

def plot_prediction_pie(prediction_probs):
    plt.figure(figsize=(8, 8))
    plt.pie(prediction_probs, labels=emotion_labels, autopct='%1.1f%%',
            startangle=140, textprops={'fontsize': 12})
    plt.title("Emotion Prediction Probabilities")
    plt.tight_layout()

    # Save to a temporary file
    temp_path = "temp_pie.png"
    plt.savefig(temp_path)
    plt.close()
    return temp_path

# -------------------------
# Main Prediction Function
# -------------------------
def predict_emotion(audio):
    if audio is None:
        return None, None, None, None, "No audio provided"

    if not model_loaded:
        return audio, None, None, None, "Model not loaded. Train the model first."

    try:
        # Load audio from Gradio
        signal, sr = librosa.load(audio, sr=SAMPLE_RATE)

        # Debug info
        print(f"Audio loaded. Shape: {signal.shape}, Sample rate: {sr}")

        # Make copies for visualization
        signal_original = signal.copy()

        # Create visualizations
        mel_img = plot_mel_spectrogram(signal_original, sr)
        mfcc_img = plot_mfcc(signal_original, sr)

        # Preprocess and extract features
        signal_processed = preprocess_audio(signal.copy(), sr)
        features = extract_features(signal_processed, sr)
        features = features.reshape(1, -1, 1)

        # Predict
        prediction = model.predict(features)[0]
        predicted_label = le.inverse_transform([np.argmax(prediction)])[0]
        pie_img = plot_prediction_pie(prediction)

        return audio, mel_img, mfcc_img, pie_img, predicted_label

    except Exception as e:
        import traceback
        print(f"Error in prediction: {e}")
        print(traceback.format_exc())
        return audio, None, None, None, f"Error: {str(e)}"

# -------------------------
# Gradio Interface
# -------------------------
with gr.Blocks(title="🎵 Speech Emotion Recognition") as demo:
    gr.Markdown("# 🎵 Emotion Recognition from Speech")
    gr.Markdown("Record or upload a short audio sample to analyze the emotion.")

    with gr.Row():
        audio_input = gr.Audio(
            sources=["microphone", "upload"],
            type="filepath",
            label="🎤 Record or Upload Speech"
        )

    with gr.Row():
        submit_btn = gr.Button("Analyze Emotion", variant="primary")

    with gr.Row():
        with gr.Column():
            audio_output = gr.Audio(label="🔊 Processed Audio")
            emotion_output = gr.Textbox(label="🎯 Predicted Emotion")

    with gr.Row():
        with gr.Column():
            mel_spec = gr.Image(label="🎼 Mel Spectrogram")
        with gr.Column():
            mfcc_plot = gr.Image(label="🎙️ MFCCs")

    with gr.Row():
        pie_chart = gr.Image(label="📊 Prediction Probabilities")

    submit_btn.click(
        fn=predict_emotion,
        inputs=[audio_input],
        outputs=[audio_output, mel_spec, mfcc_plot, pie_chart, emotion_output]
    )

    gr.Markdown("""
    ## How it works

    This application uses a trained CNN+LSTM neural network to recognize emotions from speech.

    1. **Record or upload** an audio sample
    2. Click **Analyze Emotion**
    3. View the visualizations and emotion prediction

    The model can detect: neutral, calm, happy, sad, angry, fearful, disgust, and surprised emotions.
    """)

# Launch the interface
demo.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://235b56677ed8daaedc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


##################################################################

In [ ]:
!pip install scikit-learn noisereduce librosa tensorflow matplotlib gradio noisereduce

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
# prompt: import drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow as tf
import random
from google.colab import drive

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)


# -----------------------------
# Configuration
# -----------------------------
DATASET_PATH = '/content/drive/MyDrive/SpeechFolder'  # Change this to your path
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Emotion code to label
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}



def generate_labels_csv(dataset_path):
    """Generate a CSV file containing filenames and their emotion labels."""
    data = []
    valid_files = 0
    skipped_files = 0

    for file in os.listdir(dataset_path):
        if file.endswith('.wav'):
            parts = file.split('-')
            if len(parts) >= 3:
                emotion_code = parts[2]
                emotion = emotion_map.get(emotion_code)
                if emotion:
                    data.append({'filename': file, 'emotion': emotion})
                    valid_files += 1
                else:
                    skipped_files += 1
            else:
                skipped_files += 1

    df = pd.DataFrame(data)
    csv_path = os.path.join(dataset_path, 'ravdess_labels.csv')
    df.to_csv(csv_path, index=False)

    print(f"✅ Labels CSV generated at {csv_path}!")
    print(f"📊 Files processed: {valid_files} valid, {skipped_files} skipped")

    # Print distribution of emotions
    emotion_counts = df['emotion'].value_counts()
    for emotion, count in emotion_counts.items():
        print(f"   {emotion}: {count} files")

    return df

# -----------------------------
# Preprocessing - Normalization, Denoising, Augmentation
# -----------------------------

def preprocess_audio(signal, sample_rate):
    """Apply audio preprocessing techniques to enhance signal quality."""
    # Normalize amplitude
    signal = librosa.util.normalize(signal)

    # Denoise (using Noisereduce)
    signal = nr.reduce_noise(y=signal, sr=sample_rate)

    return signal

def augment_audio(signal, sample_rate):
    """Apply data augmentation techniques to increase training data diversity."""
    # Original signal
    augmented_signals = [signal]

    # Pitch shift up
    pitch_up = librosa.effects.pitch_shift(y=signal, sr=int(sample_rate), n_steps=2)
    augmented_signals.append(pitch_up)

    # Pitch shift down
    pitch_down = librosa.effects.pitch_shift(y=signal, sr=int(sample_rate), n_steps=-2)
    augmented_signals.append(pitch_down)

    # Time stretching - faster
    stretch_fast = librosa.effects.time_stretch(y=signal, rate=1.2)
    # Ensure it fits MAX_LEN
    if len(stretch_fast) > MAX_LEN:
        stretch_fast = stretch_fast[:MAX_LEN]
    else:
        stretch_fast = np.pad(stretch_fast, (0, MAX_LEN - len(stretch_fast)))
    augmented_signals.append(stretch_fast)

    # Time stretching - slower
    stretch_slow = librosa.effects.time_stretch(y=signal, rate=0.8)
    # Ensure it fits MAX_LEN
    if len(stretch_slow) > MAX_LEN:
        stretch_slow = stretch_slow[:MAX_LEN]
    else:
        stretch_slow = np.pad(stretch_slow, (0, MAX_LEN - len(stretch_slow)))
    augmented_signals.append(stretch_slow)

    return augmented_signals

# -----------------------------
# Feature extraction: MFCC + ZCR + RMSE + Spectral Centroid + Chroma
# -----------------------------

def extract_features(signal, sr):
    """Extract audio features for emotion recognition."""
    if len(signal) > MAX_LEN:
        signal = signal[:MAX_LEN]
    else:
        signal = np.pad(signal, (0, MAX_LEN - len(signal)))

    # Extract MFCCs (increased to 20 for better feature representation)
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
    mfccs = np.mean(mfccs.T, axis=0)

    # Extract delta MFCCs (1st order derivatives)
    delta_mfccs = librosa.feature.delta(mfccs)

    # Zero-crossing rate - voice characteristics
    zcr = librosa.feature.zero_crossing_rate(y=signal)
    zcr = np.mean(zcr)

    # Root mean square energy - volume/intensity
    rmse = librosa.feature.rms(y=signal)
    rmse = np.mean(rmse)

    # Spectral centroid - brightness of sound
    spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr)
    spectral_centroid = np.mean(spectral_centroid)

    # Chroma features - tonal content
    chroma = librosa.feature.chroma_stft(y=signal, sr=sr)
    chroma = np.mean(chroma.T, axis=0)

    # Spectral contrast - difference between peaks and valleys
    spectral_contrast = librosa.feature.spectral_contrast(y=signal, sr=sr)
    spectral_contrast = np.mean(spectral_contrast.T, axis=0)

    # Combine all features
    return np.hstack([
        mfccs,            # 20 features
        delta_mfccs,      # 20 features
        zcr,              # 1 feature
        rmse,             # 1 feature
        spectral_centroid,# 1 feature
        chroma,           # 12 features
        spectral_contrast # 7 features
    ])

# -----------------------------
# Load data and extract features
# -----------------------------

def load_dataset(dataset_path, df, use_augmentation=True):
    """Load audio files, extract features, and apply augmentation if requested."""
    features, labels = [], []
    processed = 0
    total_files = len(df)

    for idx, row in df.iterrows():
        file_path = os.path.join(dataset_path, row['filename'])
        if os.path.exists(file_path):
            try:
                # Load and preprocess audio
                signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
                signal_processed = preprocess_audio(signal, sr)

                # Extract features from original processed signal
                feat = extract_features(signal_processed, sr)
                features.append(feat)
                labels.append(row['emotion'])

                # Apply augmentation if requested
                if use_augmentation:
                    augmented_signals = augment_audio(signal_processed, sr)
                    # Skip first signal as it's the original one
                    for aug_signal in augmented_signals[1:]:
                        aug_feat = extract_features(aug_signal, sr)
                        features.append(aug_feat)
                        labels.append(row['emotion'])

                processed += 1
                if processed % 10 == 0:
                    print(f"Processed {processed}/{total_files} files...")

            except Exception as e:
                print(f"⚠️ Failed to process {file_path}: {e}")

    print(f"✅ Processed {processed} files, created {len(features)} feature vectors")
    return np.array(features), np.array(labels)

# -----------------------------
# Build an improved model: CNN + LSTM with attention
# -----------------------------

def build_model(input_shape, num_classes):
    """Build a CNN-LSTM model architecture with attention mechanism."""
    model = Sequential()

    # First Conv Block
    model.add(Conv1D(64, kernel_size=5, strides=1, padding='same', activation='relu', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    # Second Conv Block
    model.add(Conv1D(128, kernel_size=3, strides=1, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    # Third Conv Block
    model.add(Conv1D(256, kernel_size=3, strides=1, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    # LSTM layer
    model.add(LSTM(128, return_sequences=False))
    model.add(Dropout(0.4))

    # Dense layers
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    # Compile with Adam optimizer and categorical crossentropy loss
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=['accuracy']
    )

    return model

# -----------------------------
# Main pipeline
# -----------------------------

def main():
    print("🚀 Starting Speech Emotion Recognition Model Training")

    # Generate the labels CSV if it doesn't exist
    csv_path = os.path.join(DATASET_PATH, 'ravdess_labels.csv')
    if os.path.exists(csv_path):
        print(f"📄 Loading existing labels from {csv_path}")
        df = pd.read_csv(csv_path)
    else:
        print("📝 Generating labels CSV...")
        df = generate_labels_csv(DATASET_PATH)

    # Check if DataFrame is empty
    if df.empty:
        print("❌ No valid audio files found. Please check your dataset path and file naming convention.")
        return

    print("🔍 Loading and processing data...")
    X, y = load_dataset(DATASET_PATH, df, use_augmentation=True)

    # Check if any data was loaded
    if len(X) == 0:
        print("❌ No audio files processed. Please check DATASET_PATH and file naming.")
        return

    print(f"✅ Dataset loaded: {X.shape[0]} samples")

    # Encode labels
    print("🏷️ Encoding labels...")
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    y_categorical = to_categorical(y_encoded)
    print(f"✅ Label mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")

    # Reshape features for CNN input: (samples, timesteps, features)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    # Split data into train and test sets
    print("🔪 Splitting data into train and test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
    )
    print(f"✅ Train set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")

    # Build and train the model
    print("🧠 Building model architecture...")
    model = build_model(input_shape=(X.shape[1], 1), num_classes=y_categorical.shape[1])
    model.summary()

    # Set up callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001),
        ModelCheckpoint('best_emotion_model.h5', monitor='val_accuracy', save_best_only=True, mode='max')
    ]

    # Train the model
    print("🏋️ Training model...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        batch_size=32,
        epochs=100,
        callbacks=callbacks,
        verbose=1
    )

    print("✅ Training complete!")

    # Evaluate the model
    test_loss, test_acc = model.evaluate(X_test, y_test)
    print(f"📊 Test accuracy: {test_acc:.4f}")
    print(f"📊 Test loss: {test_loss:.4f}")

    # Save model
    model.save("emotion_recognition_model.h5")
    print("📁 Model saved as emotion_recognition_model.h5")

    # Save label encoder classes
    np.save('emotion_classes.npy', le.classes_)
    print("📁 Label encoder classes saved as emotion_classes.npy")

    # Predict on a test sample
    if not df.empty:
        test_path = os.path.join(DATASET_PATH, df['filename'].iloc[0])
        signal, sr = librosa.load(test_path, sr=SAMPLE_RATE)
        signal_processed = preprocess_audio(signal, sr)
        test_feat = extract_features(signal_processed, sr).reshape(1, -1, 1)
        pred = model.predict(test_feat)
        pred_label = le.inverse_transform([np.argmax(pred)])
        print(f"🎤 Sample Prediction Test:")
        print(f"   File: {df['filename'].iloc[0]}")
        print(f"   True emotion: {df['emotion'].iloc[0]}")
        print(f"   Predicted emotion: {pred_label[0]}")
        print(f"   Confidence: {np.max(pred):.2f}")

if __name__ == "__main__":
    main()

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
🚀 Starting Speech Emotion Recognition Model Training
📄 Loading existing labels from /content/drive/MyDrive/SpeechFolder/ravdess_labels.csv
🔍 Loading and processing data...
Processed 10/1036 files...
Processed 20/1036 files...
Processed 30/1036 files...
Processed 40/1036 files...
Processed 50/1036 files...
Processed 60/1036 files...
Processed 70/1036 files...
Processed 80/1036 files...
Processed 90/1036 files...
Processed 100/1036 files...
Processed 110/1036 files...
Processed 120/1036 files...
Processed 130/10

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 62, 64)         │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 62, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 31, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 31, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 31, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 31, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 15, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 15, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 348,616 (1.33 MB)

...

130/130 ━━━━━━━━━━━━━━━━━━━━ 7s 50ms/step - accuracy: 0.5286 - loss: 1.3006 - val_accuracy: 0.5328 - val_loss: 1.3029 - learning_rate: 5.0000e-04
Epoch 89/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.5363 - loss: 1.2870 - val_accuracy: 0.5145 - val_loss: 1.3508 - learning_rate: 5.0000e-04
Epoch 90/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5334 - loss: 1.2711

130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.5334 - loss: 1.2710 - val_accuracy: 0.5376 - val_loss: 1.3182 - learning_rate: 5.0000e-04
Epoch 91/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.5527 - loss: 1.2657 - val_accuracy: 0.5328 - val_loss: 1.3186 - learning_rate: 5.0000e-04
Epoch 92/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 6s 43ms/step - accuracy: 0.5592 - loss: 1.2743 - val_accuracy: 0.5280 - val_loss: 1.3326 - learning_rate: 5.0000e-04
Epoch 93/100
129/130 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5568 - loss: 1.2415

130/130 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.5567 - loss: 1.2414 - val_accuracy: 0.5763 - val_loss: 1.2076 - learning_rate: 5.0000e-04
Epoch 94/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.5565 - loss: 1.2317 - val_accuracy: 0.5125 - val_loss: 1.3665 - learning_rate: 5.0000e-04
Epoch 95/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.5624 - loss: 1.2207 - val_accuracy: 0.5222 - val_loss: 1.3449 - learning_rate: 5.0000e-04
Epoch 96/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - accuracy: 0.5699 - loss: 1.2190 - val_accuracy: 0.5125 - val_loss: 1.3297 - learning_rate: 5.0000e-04
Epoch 97/100
129/130 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5729 - loss: 1.2123

130/130 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.5727 - loss: 1.2122 - val_accuracy: 0.5927 - val_loss: 1.1513 - learning_rate: 5.0000e-04
Epoch 98/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.5672 - loss: 1.1953 - val_accuracy: 0.5203 - val_loss: 1.3643 - learning_rate: 5.0000e-04
Epoch 99/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 10s 43ms/step - accuracy: 0.5772 - loss: 1.1918 - val_accuracy: 0.5869 - val_loss: 1.1753 - learning_rate: 5.0000e-04
Epoch 100/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.5841 - loss: 1.1714 - val_accuracy: 0.5048 - val_loss: 1.3939 - learning_rate: 5.0000e-04
✅ Training complete!
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5952 - loss: 1.1532


📊 Test accuracy: 0.5927
📊 Test loss: 1.1513
📁 Model saved as emotion_recognition_model.h5
📁 Label encoder classes saved as emotion_classes.npy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step
🎤 Sample Prediction Test:
   File: 03-01-05-02-02-02-01.wav
   True emotion: angry
   Predicted emotion: angry
   Confidence: 0.51


In [ ]:
import gradio as gr
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import noisereduce as nr
from tensorflow.keras.models import load_model
from PIL import Image
import io
import os
import time
import warnings
warnings.filterwarnings('ignore')

# -------------------------
# Configuration
# -------------------------
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Install required packages (uncomment if running in Colab)
# !pip install gradio librosa matplotlib tensorflow noisereduce

# -------------------------
# Load model and emotion classes
# -------------------------
def load_emotion_model():
    """Load the emotion recognition model with proper error handling"""
    try:
        model = load_model("/content/emotion_recognition_model.h5")
        print("✅ Model loaded successfully!")

        # Try to load the classes from the saved file
        if os.path.exists('emotion_classes.npy'):
            emotion_labels = np.load('emotion_classes.npy', allow_pickle=True)
            print(f"✅ Emotion classes loaded: {emotion_labels}")
        else:
            # Default emotion labels if file not found
            emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
            print("⚠️ Using default emotion labels")

        return model, emotion_labels

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("⚠️ Creating a placeholder model for demonstration")

        # Create a dummy model for demonstration purposes
        from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
        from tensorflow.keras.models import Model

        # More realistic model architecture
        inputs = Input(shape=(62, 1))
        x = LSTM(128, return_sequences=True, dropout=0.2)(inputs)
        x = LSTM(64, dropout=0.2)(x)
        x = Dense(32, activation='relu')(x)
        x = Dropout(0.3)(x)
        outputs = Dense(8, activation='softmax')(x)

        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

        # Default emotion labels
        emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
        print("⚠️ Using placeholder model and default emotion labels")

        return model, emotion_labels

# Load model and labels globally
model, emotion_labels = load_emotion_model()

# -------------------------
# Audio Processing Functions
# -------------------------
def preprocess_audio(signal, sr):
    """Apply comprehensive preprocessing to the audio signal"""
    try:
        # Ensure the sample rate is correct
        if sr != SAMPLE_RATE:
            signal = librosa.resample(signal, orig_sr=sr, target_sr=SAMPLE_RATE)
            sr = SAMPLE_RATE

        # Remove silence from beginning and end
        signal, _ = librosa.effects.trim(signal, top_db=20)

        # Normalize audio to [-1, 1] range
        signal = librosa.util.normalize(signal)

        # Apply noise reduction
        signal = nr.reduce_noise(y=signal, sr=sr, stationary=False)

        # Apply pre-emphasis filter to balance the frequency spectrum
        signal = np.append(signal[0], signal[1:] - 0.97 * signal[:-1])

        return signal, sr
    except Exception as e:
        print(f"Warning in preprocessing: {e}")
        # Return original signal if preprocessing fails
        return librosa.util.normalize(signal), sr

def extract_comprehensive_features(signal, sr):
    """Extract comprehensive audio features for emotion recognition"""
    try:
        # Handle audio length - pad or truncate to fixed length
        if len(signal) > MAX_LEN:
            signal = signal[:MAX_LEN]
        else:
            signal = np.pad(signal, (0, MAX_LEN - len(signal)))

        features = []

        # 1. MFCCs (Mel-frequency cepstral coefficients) - 20 features
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
        mfccs_mean = np.mean(mfccs.T, axis=0)
        mfccs_std = np.std(mfccs.T, axis=0)
        features.extend(mfccs_mean)
        features.extend(mfccs_std)  # Add standard deviation for more robustness

        # 2. Delta MFCCs (1st order derivatives) - 20 features
        delta_mfccs = librosa.feature.delta(mfccs)
        delta_mfccs_mean = np.mean(delta_mfccs.T, axis=0)
        features.extend(delta_mfccs_mean)

        # 3. Delta-Delta MFCCs (2nd order derivatives) - 20 features
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        delta2_mfccs_mean = np.mean(delta2_mfccs.T, axis=0)
        features.extend(delta2_mfccs_mean)

        # 4. Zero-crossing rate - voice characteristics
        zcr = librosa.feature.zero_crossing_rate(y=signal, frame_length=2048, hop_length=512)
        features.append(np.mean(zcr))
        features.append(np.std(zcr))

        # 5. Root mean square energy - volume/intensity
        rmse = librosa.feature.rms(y=signal, frame_length=2048, hop_length=512)
        features.append(np.mean(rmse))
        features.append(np.std(rmse))

        # 6. Spectral centroid - brightness of sound
        spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_centroid))
        features.append(np.std(spectral_centroid))

        # 7. Spectral rolloff - frequency below which 85% of energy is contained
        spectral_rolloff = librosa.feature.spectral_rolloff(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_rolloff))
        features.append(np.std(spectral_rolloff))

        # 8. Spectral bandwidth - width of the frequency band
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_bandwidth))
        features.append(np.std(spectral_bandwidth))

        # Convert to numpy array and ensure we have exactly 62 features
        features = np.array(features)

        # Pad or truncate to exactly 62 features to match model input
        if len(features) < 62:
            features = np.pad(features, (0, 62 - len(features)))
        elif len(features) > 62:
            features = features[:62]

        print(f"✅ Extracted {len(features)} audio features")
        return features

    except Exception as e:
        print(f"❌ Error in feature extraction: {e}")
        # Return dummy features if extraction fails
        return np.zeros(62)  # Match expected feature count

# -------------------------
# Enhanced Visualization Functions
# -------------------------
def create_waveform_plot(signal, sr, title="Audio Waveform"):
    """Generate enhanced waveform plot and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

        # Time axis
        time_axis = np.linspace(0, len(signal) / sr, len(signal))

        # Original waveform
        ax1.plot(time_axis, signal, color='#00d4ff', linewidth=0.8, alpha=0.8)
        ax1.fill_between(time_axis, signal, alpha=0.3, color='#00d4ff')
        ax1.set_title(f'{title} - Time Domain', fontsize=12, color='white', pad=10)
        ax1.set_xlabel('Time (seconds)', color='white')
        ax1.set_ylabel('Amplitude', color='white')
        ax1.grid(True, alpha=0.3)
        ax1.tick_params(colors='white')

        # Spectrogram
        D = librosa.amplitude_to_db(np.abs(librosa.stft(signal)), ref=np.max)
        img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz',
                                      ax=ax2, cmap='plasma', alpha=0.8)
        ax2.set_title('Spectrogram - Frequency Domain', fontsize=12, color='white', pad=10)
        ax2.tick_params(colors='white')
        plt.colorbar(img, ax=ax2, format='%+2.0f dB')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating waveform plot: {e}")
        return create_error_image("Error generating waveform visualization")

def create_mel_spectrogram_plot(signal, sr):
    """Generate enhanced Mel Spectrogram visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # Mel Spectrogram
        S = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128,
                                         n_fft=2048, hop_length=512, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)

        img1 = librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel',
                                       fmax=8000, ax=ax1, cmap='viridis')
        ax1.set_title('Mel Spectrogram', fontsize=14, color='white', pad=15)
        ax1.tick_params(colors='white')
        plt.colorbar(img1, ax=ax1, format='%+2.0f dB')

        # Mel filter bank visualization
        mel_filters = librosa.filters.mel(sr=sr, n_fft=2048, n_mels=10, fmax=8000)
        ax2.plot(mel_filters.T, alpha=0.8, linewidth=1.5)
        ax2.set_title('Mel Filter Bank (First 10 filters)', fontsize=14, color='white', pad=15)
        ax2.set_xlabel('Frequency Bins', color='white')
        ax2.set_ylabel('Filter Response', color='white')
        ax2.grid(True, alpha=0.3)
        ax2.tick_params(colors='white')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating mel spectrogram: {e}")
        return create_error_image("Error generating mel spectrogram")

def create_mfcc_plot(signal, sr):
    """Generate enhanced MFCC visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

        # MFCCs
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
        img1 = librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax1, cmap='coolwarm')
        ax1.set_title('MFCCs (Mel-frequency Cepstral Coefficients)', fontsize=12, color='white')
        ax1.set_ylabel('MFCC Coefficients', color='white')
        ax1.tick_params(colors='white')
        plt.colorbar(img1, ax=ax1)

        # Delta MFCCs
        delta_mfccs = librosa.feature.delta(mfccs)
        img2 = librosa.display.specshow(delta_mfccs, sr=sr, x_axis='time', ax=ax2, cmap='coolwarm')
        ax2.set_title('Delta MFCCs (1st derivatives)', fontsize=12, color='white')
        ax2.set_ylabel('Delta MFCC', color='white')
        ax2.tick_params(colors='white')
        plt.colorbar(img2, ax=ax2)

        # Delta-Delta MFCCs
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        img3 = librosa.display.specshow(delta2_mfccs, sr=sr, x_axis='time', ax=ax3, cmap='coolwarm')
        ax3.set_title('Delta-Delta MFCCs (2nd derivatives)', fontsize=12, color='white')
        ax3.set_ylabel('Delta-Delta MFCC', color='white')
        ax3.set_xlabel('Time (s)', color='white')
        ax3.tick_params(colors='white')
        plt.colorbar(img3, ax=ax3)

        # MFCC statistics
        mfcc_mean = np.mean(mfccs.T, axis=0)
        mfcc_std = np.std(mfccs.T, axis=0)

        x_pos = np.arange(len(mfcc_mean))
        ax4.bar(x_pos - 0.2, mfcc_mean, 0.4, label='Mean', alpha=0.8, color='#ff6b6b')
        ax4.bar(x_pos + 0.2, mfcc_std, 0.4, label='Std Dev', alpha=0.8, color='#4ecdc4')
        ax4.set_title('MFCC Statistics', fontsize=12, color='white')
        ax4.set_xlabel('MFCC Coefficient Index', color='white')
        ax4.set_ylabel('Value', color='white')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        ax4.tick_params(colors='white')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating MFCC plot: {e}")
        return create_error_image("Error generating MFCC visualization")

def create_prediction_plot(prediction_probs, feature_importance=None):
    """Generate comprehensive prediction visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

        # 1. Horizontal bar chart of predictions
        sorted_indices = np.argsort(prediction_probs)
        sorted_probs = prediction_probs[sorted_indices]
        sorted_labels = emotion_labels[sorted_indices]

        colors = plt.cm.viridis(np.linspace(0, 1, len(sorted_probs)))
        bars = ax1.barh(sorted_labels, sorted_probs * 100, color=colors, alpha=0.8)
        ax1.set_xlabel('Probability (%)', color='white', fontsize=12)
        ax1.set_title('Emotion Prediction Probabilities', color='white', fontsize=14, pad=15)
        ax1.tick_params(colors='white')
        ax1.grid(True, alpha=0.3, axis='x')

        # Add percentage labels on bars
        for bar, prob in zip(bars, sorted_probs):
            width = bar.get_width()
            ax1.text(width + 1, bar.get_y() + bar.get_height()/2,
                    f'{prob*100:.1f}%', ha='left', va='center', color='white', fontsize=10)

        # 2. Pie chart for top emotions
        top_n = 5
        top_indices = np.argsort(prediction_probs)[::-1][:top_n]
        top_probs = prediction_probs[top_indices]
        top_labels = emotion_labels[top_indices]

        # Add "others" category if needed
        others_prob = 1 - np.sum(top_probs)
        if others_prob > 0.01:  # Only add if significant
            top_probs = np.append(top_probs, others_prob)
            top_labels = np.append(top_labels, 'Others')

        colors_pie = plt.cm.Set3(np.linspace(0, 1, len(top_probs)))
        wedges, texts, autotexts = ax2.pie(top_probs, labels=top_labels, autopct='%1.1f%%',
                                          startangle=90, colors=colors_pie,
                                          textprops={'color': 'white', 'fontsize': 10})
        ax2.set_title('Top Emotion Predictions', color='white', fontsize=14, pad=15)

        # 3. Confidence analysis
        max_prob = np.max(prediction_probs)
        entropy = -np.sum(prediction_probs * np.log2(prediction_probs + 1e-10))
        max_entropy = np.log2(len(emotion_labels))
        normalized_entropy = entropy / max_entropy

        confidence_metrics = ['Max Probability', 'Certainty\n(1 - Entropy)', 'Top-2 Gap']
        top2_gap = sorted_probs[-1] - sorted_probs[-2] if len(sorted_probs) > 1 else max_prob
        confidence_values = [max_prob, 1 - normalized_entropy, top2_gap]

        bars3 = ax3.bar(confidence_metrics, confidence_values,
                       color=['#ff6b6b', '#4ecdc4', '#45b7d1'], alpha=0.8)
        ax3.set_title('Prediction Confidence Metrics', color='white', fontsize=14, pad=15)
        ax3.set_ylabel('Score', color='white')
        ax3.tick_params(colors='white')
        ax3.grid(True, alpha=0.3, axis='y')

        # Add value labels on bars
        for bar, value in zip(bars3, confidence_values):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{value:.3f}', ha='center', va='bottom', color='white', fontsize=11)

        # 4. Prediction distribution
        ax4.plot(emotion_labels, prediction_probs * 100, 'o-',
                color='#ffa726', linewidth=2, markersize=8, alpha=0.8)
        ax4.fill_between(range(len(emotion_labels)), prediction_probs * 100,
                        alpha=0.3, color='#ffa726')
        ax4.set_title('Prediction Distribution', color='white', fontsize=14, pad=15)
        ax4.set_ylabel('Probability (%)', color='white')
        ax4.set_xlabel('Emotions', color='white')
        ax4.tick_params(colors='white', axis='x', rotation=45)
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating prediction plot: {e}")
        return create_error_image("Error generating prediction visualization")

def create_error_image(error_message):
    """Create an error image when visualization fails"""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, f"❌ {error_message}", transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='red',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.8))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close()
    buf.seek(0)

    return Image.open(buf)

# -------------------------
# Main Prediction Function
# -------------------------
def predict_emotion_comprehensive(audio):
    """Process audio and predict emotion with comprehensive analysis"""

    if audio is None:
        return (
            None,  # audio playback
            create_error_image("No audio provided"),  # waveform
            create_error_image("No audio provided"),  # mel spectrogram
            create_error_image("No audio provided"),  # mfcc
            create_error_image("No audio provided"),  # prediction
            "❌ No audio provided. Please record or upload audio."  # text result
        )

    try:
        print("🎵 Loading audio file...")
        # Load audio with librosa
        signal, sr = librosa.load(audio, sr=None)
        original_signal = signal.copy()
        original_sr = sr

        print(f"✅ Audio loaded: {len(signal)/sr:.2f}s, {sr}Hz sample rate")

        # Generate waveform plot (before preprocessing)
        print("📊 Generating waveform visualization...")
        waveform_img = create_waveform_plot(original_signal, original_sr, "Original Waveform")

        # Preprocess audio
        print("🔧 Preprocessing audio...")
        processed_signal, sr = preprocess_audio(signal, sr)

        # Generate mel spectrogram
        print("🎼 Generating mel spectrogram...")
        mel_img = create_mel_spectrogram_plot(processed_signal, sr)

        # Generate MFCC visualization
        print("📈 Generating MFCC analysis...")
        mfcc_img = create_mfcc_plot(processed_signal, sr)

        # Extract features
        print("🔍 Extracting audio features...")
        features = extract_comprehensive_features(processed_signal, sr)

        # Reshape for model input (samples, timesteps, features)
        features_reshaped = features.reshape(1, features.shape[0], 1)

        # Make prediction
        print("🧠 Running emotion prediction...")
        prediction = model.predict(features_reshaped, verbose=0)[0]

        # Get the predicted emotion
        predicted_emotion = emotion_labels[np.argmax(prediction)]
        confidence = np.max(prediction) * 100

        # Generate comprehensive prediction visualization
        print("📊 Generating prediction analysis...")
        prediction_img = create_prediction_plot(prediction)

        # Create detailed result text
        result_lines = [
            f"🎭 **Predicted Emotion: {predicted_emotion.upper()}**",
            f"🎯 **Confidence: {confidence:.1f}%**",
            f"⏱️ **Audio Duration: {len(original_signal)/original_sr:.2f} seconds**",
            f"🔊 **Sample Rate: {original_sr} Hz**",
            f"📊 **Features Extracted: {len(features)}**",
            "",
            "**Top 3 Predictions:**"
        ]

        # Add top 3 predictions
        top_3_indices = np.argsort(prediction)[::-1][:3]
        for i, idx in enumerate(top_3_indices, 1):
            emotion = emotion_labels[idx]
            prob = prediction[idx] * 100
            result_lines.append(f"{i}. {emotion.capitalize()}: {prob:.1f}%")

        result_text = "\n".join(result_lines)

        print("✅ Analysis complete!")

        return (
            audio,           # audio playback
            waveform_img,    # waveform plot
            mel_img,         # mel spectrogram
            mfcc_img,        # mfcc plot
            prediction_img,  # prediction analysis
            result_text      # text result
        )

    except Exception as e:
        error_msg = f"❌ Error processing audio: {str(e)}"
        print(error_msg)

        return (
            None,  # audio playback
            create_error_image("Processing failed"),  # waveform
            create_error_image("Processing failed"),  # mel spectrogram
            create_error_image("Processing failed"),  # mfcc
            create_error_image("Processing failed"),  # prediction
            error_msg  # text result
        )

# -------------------------
# Enhanced Gradio Interface
# -------------------------
def create_interface():
    """Create the enhanced Gradio interface"""

    # Custom CSS for better styling
    custom_css = """
    .gradio-container {
        max-width: 1200px !important;
    }

    .main-header {
        text-align: center;
        background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-size: 2.5em;
        font-weight: bold;
        margin-bottom: 0.5em;
    }

    .description {
        text-align: center;
        font-size: 1.1em;
        color: #666;
        margin-bottom: 2em;
        max-width: 800px;
        margin-left: auto;
        margin-right: auto;
    }

    .result-box {
        background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
        border-radius: 10px;
        padding: 20px;
        margin: 10px 0;
    }
    """

    with gr.Blocks(
        title="🎭 Advanced Speech Emotion Recognition",
        theme=gr.themes.Soft(),
        css=custom_css
    ) as interface:

        gr.HTML("""
        <div class="main-header">🎭 Advanced Speech Emotion Recognition</div>
        <div class="description">
            Upload or record audio to analyze emotional content using advanced deep learning techniques.
            This system analyzes multiple audio features including MFCCs, spectral characteristics,
            and temporal dynamics to predict emotions with high accuracy.
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=1):
                # Audio input section
                gr.Markdown("### 🎤 Audio Input")
                audio_input = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Record or Upload Speech (3-10 seconds recommended)",
                    show_download_button=True
                )

                # Analysis button
                analyze_btn = gr.Button(
                    "🚀 Analyze Emotion",
                    variant="primary",
                    size="lg",
                    scale=1
                )

                # Audio playback
                gr.Markdown("### 🔊 Audio Playback")
                audio_output = gr.Audio(
                    label="Processed Audio",
                    show_download_button=True
                )

            with gr.Column(scale=2):
                # Results section
                gr.Markdown("### 📊 Analysis Results")
                result_text = gr.Markdown(
                    "Ready for analysis. Upload or record audio and click 'Analyze Emotion'.",
                    elem_classes=["result-box"]
                )

        # Visualization tabs
        gr.Markdown("### 📈 Detailed Visualizations")
        with gr.Tabs():
            with gr.TabItem("🌊 Audio Waveform"):
                waveform_plot = gr.Image(
                    label="Audio Waveform & Spectrogram",
                    type="pil"
                )

            with gr.TabItem("🎼 Mel Spectrogram"):
                mel_plot = gr.Image(
                    label="Mel Spectrogram Analysis",
                    type="pil"
                )

            with gr.TabItem("📊 MFCC Features"):
                mfcc_plot = gr.Image(
                    label="MFCC Feature Analysis",
                    type="pil"
                )

            with gr.TabItem("🎯 Predictions"):
                prediction_plot = gr.Image(
                    label="Emotion Prediction Analysis",
                    type="pil"
                )

        # Example section
        gr.Markdown("### 🎵 Try These Examples")
        with gr.Row():
            gr.Examples(
                examples=[
                    # Add paths to your example audio files here if you have any
                    # ["path/to/happy_example.wav"],
                    # ["path/to/sad_example.wav"],
                    # ["path/to/angry_example.wav"],
                ],
                inputs=audio_input,
                outputs=[audio_output, waveform_plot, mel_plot, mfcc_plot, prediction_plot, result_text],
                fn=predict_emotion_comprehensive,
                cache_examples=False
            )

        # Information section
        with gr.Accordion("ℹ️ About This System", open=False):
            gr.Markdown("""
            ### How It Works:
            1. **Audio Preprocessing**: Noise reduction, normalization, and trimming
            2. **Feature Extraction**: 62 comprehensive audio features including:
               - MFCC coefficients (20 features)
               - Delta and Delta-Delta MFCCs (40 features)
               - Spectral features (centroid, rolloff, bandwidth)
               - Temporal features (zero-crossing rate, RMS energy)
            3. **Deep Learning Model**: LSTM-based neural network for emotion classification
            4. **Emotion Classes**: Neutral, Calm, Happy, Sad, Angry, Fearful, Disgust, Surprised

            ### Supported Audio Formats:
            - WAV, MP3, FLAC, M4A, and other common formats
            - Recommended: 3-10 seconds of clear speech
            - Sample rate: automatically converted to 22.05 kHz

            ### Tips for Best Results:
            - Use clear, unambiguous emotional speech
            - Minimize background noise
            - Speak naturally and expressively
            - Audio length between 3-10 seconds works best
            """)

        # Connect the button to the function
        analyze_btn.click(
            fn=predict_emotion_comprehensive,
            inputs=[audio_input],
            outputs=[audio_output, waveform_plot, mel_plot, mfcc_plot, prediction_plot, result_text],
            show_progress=True
        )

        # Also trigger on audio input change
        audio_input.change(
            fn=predict_emotion_comprehensive,
            inputs=[audio_input],
            outputs=[audio_output, waveform_plot, mel_plot, mfcc_plot, prediction_plot, result_text],
            show_progress=True
        )

    return interface

# -------------------------
# Launch the application
# -------------------------
if __name__ == "__main__":
    print("🚀 Starting Speech Emotion Recognition System...")
    print(f"📊 Model loaded with {len(emotion_labels)} emotion classes: {list(emotion_labels)}")

    # Create and launch the interface
    interface = create_interface()

    # Launch with sharing enabled for public access
    interface.launch(
        server_name="0.0.0.0",  # Allow external connections
        server_port=7860,       # Standard Gradio port
        share=True,             # Create public link
        debug=True,             # Enable debug mode
        show_error=True         # Show errors in interface
    )

✅ Model loaded successfully!
⚠️ Using default emotion labels
🚀 Starting Speech Emotion Recognition System...
📊 Model loaded with 8 emotion classes: [np.str_('neutral'), np.str_('calm'), np.str_('happy'), np.str_('sad'), np.str_('angry'), np.str_('fearful'), np.str_('disgust'), np.str_('surprised')]
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8836e08c7ceb739a1b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎵 Loading audio file...
✅ Audio loaded: 3.30s, 48000Hz sample rate
📊 Generating waveform visualization...
🔧 Preprocessing audio...
🎼 Generating mel spectrogram...
📈 Generating MFCC analysis...
🔍 Extracting audio features...
✅ Extracted 62 audio features
🧠 Running emotion prediction...
📊 Generating prediction analysis...
✅ Analysis complete!
🎵 Loading audio file...
✅ Audio loaded: 3.30s, 48000Hz sample rate
📊 Generating waveform visualization...
🔧 Preprocessing audio...
🎼 Generating mel spectrogram...
📈 Generating MFCC analysis...
🔍 Extracting audio features...
✅ Extracted 62 audio features
🧠 Running emotion prediction...
📊 Generating prediction analysis...
✅ Analysis complete!
🎵 Loading audio file...
✅ Audio loaded: 3.50s, 48000Hz sample rate
📊 Generating waveform visualization...
🔧 Preprocessing audio...
🎼 Generating mel spectrogram...
📈 Generating MFCC analysis...
🔍 Extracting audio features...
✅ Extracted 62 audio features
🧠 Running emotion prediction...
📊 Generating prediction anal

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.2 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import random
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# -----------------------------
# Configuration
# -----------------------------
DATASET_PATH = '/content/drive/MyDrive/SpeechFolder'  # Change this to your path
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Emotion code to label
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

# -----------------------------
# Install necessary libraries
# -----------------------------
!pip install noisereduce

# -----------------------------
# Extract emotion labels from filenames
# -----------------------------

def generate_labels_csv(dataset_path):
    data = []
    # The original code iterated through subfolders, but the file structure in the second block
    # of code suggests files are directly in DATASET_PATH. Adopting the second block's approach.
    for file in os.listdir(dataset_path):
        if file.endswith('.wav'):
            parts = file.split('-')
            if len(parts) >= 3:
                emotion_code = parts[2]
                emotion = emotion_map.get(emotion_code)
                if emotion:
                    # In the second block, 'filename' just stored the file name, not the full path
                    data.append({'filename': file, 'emotion': emotion})
    df = pd.DataFrame(data)
    csv_path = os.path.join(dataset_path, 'ravdess_labels.csv')
    df.to_csv(csv_path, index=False)
    print(f"✅ Labels CSV generated at {csv_path}!")

# -----------------------------
# Preprocessing - Normalization, Denoising, Augmentation
# -----------------------------

def preprocess_audio(signal, sample_rate):
    # Normalize amplitude
    signal = librosa.util.normalize(signal)

    # Denoise (using Noisereduce)
    signal = nr.reduce_noise(y=signal, sr=sample_rate)

    # Augmentation: Random pitch shift
    if random.random() > 0.5:
        # Ensure sample_rate is an integer for pitch_shift
        signal = librosa.effects.pitch_shift(y=signal, sr=int(sample_rate), n_steps=random.randint(-2, 2))

    # Augmentation: Time stretching
    if random.random() > 0.5:
        # Ensure rate is within valid range and time_stretch expects float
        signal = librosa.effects.time_stretch(y=signal, rate=random.uniform(0.8, 1.2))

    return signal

# -----------------------------
# Feature extraction: MFCC + ZCR + RMSE
# -----------------------------

def extract_features(file_path):
    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    if len(signal) > MAX_LEN:
        signal = signal[:MAX_LEN]
    else:
        signal = np.pad(signal, (0, MAX_LEN - len(signal)))

    # Apply preprocessing steps to the signal
    # Pass the correct sample rate used during loading
    signal = preprocess_audio(signal, sr)

    # Ensure sr is passed to librosa feature extraction functions
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    mfccs = np.mean(mfccs.T, axis=0)

    zcr = librosa.feature.zero_crossing_rate(y=signal) # zcr also needs y argument
    zcr = np.mean(zcr)

    rmse = librosa.feature.rms(y=signal)
    rmse = np.mean(rmse)

    return np.hstack([mfccs, zcr, rmse])

# -----------------------------
# Load data and extract features
# -----------------------------

# Modified load_dataset to read from the generated CSV
def load_dataset(dataset_path, df): # Added df as an argument
    features, labels = [], []

    for _, row in df.iterrows():
        file_path = os.path.join(dataset_path, row['filename'])
        if os.path.exists(file_path):
            try:
                feat = extract_features(file_path)
                features.append(feat)
                labels.append(row['emotion'])
            except Exception as e:
                print(f"⚠️ Failed to process {file_path}: {e}")

    return np.array(features), np.array(labels)

# -----------------------------
# Build the CNN + LSTM model
# -----------------------------

def build_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(64))
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# -----------------------------
# Main pipeline
# -----------------------------

def main():
    # Generate the labels CSV first if it doesn't exist
    csv_path = os.path.join(DATASET_PATH, 'ravdess_labels.csv')
    if not os.path.exists(csv_path):
        print("Generating labels CSV...")
        generate_labels_csv(DATASET_PATH)

    # Load the DataFrame into the main scope
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: Labels CSV not found at {csv_path}. Please ensure the DATASET_PATH is correct and contains audio files.")
        return
    except pd.errors.EmptyDataError:
        print(f"Error: Labels CSV at {csv_path} is empty. Please ensure your DATASET_PATH contains valid audio files with correct naming convention.")
        return


    print("🔍 Loading and processing data...")
    # Pass the loaded DataFrame to load_dataset
    X, y = load_dataset(DATASET_PATH, df)

    # Check if any data was loaded
    if len(X) == 0:
        print("❌ No audio files processed. Please check DATASET_PATH and file naming.")
        return

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    y_categorical = to_categorical(y_encoded)

    X = X[..., np.newaxis]  # Add channel dimension
    X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded) # Added stratify

    print("🧠 Training model...")
    model = build_model(input_shape=(X.shape[1], 1), num_classes=y_categorical.shape[1])
    model.summary()
    model.fit(X_train, y_train, validation_data=(X_test, y_test), batch_size=32, epochs=50)

    print("✅ Training complete!")

    # Save model
    model.save("emotion_recognition_model.h5")
    print("📁 Model saved as emotion_recognition_model.h5")

    # Test prediction
    # Ensure df is not empty before trying to access its elements
    if not df.empty:
        test_path = os.path.join(DATASET_PATH, df['filename'].iloc[0])  # Use the first file for testing
        test_feat = extract_features(test_path).reshape(1, -1, 1)
        pred = model.predict(test_feat)
        pred_label = le.inverse_transform([np.argmax(pred)])
        print(f"🎤 Predicted Emotion: {pred_label[0]}")
    else:
        print("⚠️ Cannot perform test prediction: DataFrame is empty.")


if __name__ == "__main__":
    main()

In [ ]:
import gradio as gr
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import noisereduce as nr
from tensorflow.keras.models import load_model
from PIL import Image
import io
import os
import warnings
warnings.filterwarnings('ignore')

# -------------------------
# Configuration
# -------------------------
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# -------------------------
# Load model and emotion classes
# -------------------------
def load_emotion_model():
    """Load the emotion recognition model with proper error handling"""
    try:
        model = load_model("/content/emotion_recognition_model.h5")
        print("✅ Model loaded successfully!")

        # Try to load the classes from the saved file
        if os.path.exists('emotion_classes.npy'):
            emotion_labels = np.load('emotion_classes.npy', allow_pickle=True)
            print(f"✅ Emotion classes loaded: {emotion_labels}")
        else:
            # Default emotion labels if file not found
            emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
            print("⚠️ Using default emotion labels")

        return model, emotion_labels

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("⚠️ Creating a placeholder model for demonstration")

        # Create a dummy model for demonstration purposes
        from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
        from tensorflow.keras.models import Model

        # More realistic model architecture
        inputs = Input(shape=(62, 1))
        x = LSTM(128, return_sequences=True, dropout=0.2)(inputs)
        x = LSTM(64, dropout=0.2)(x)
        x = Dense(32, activation='relu')(x)
        x = Dropout(0.3)(x)
        outputs = Dense(8, activation='softmax')(x)

        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

        # Default emotion labels
        emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
        print("⚠️ Using placeholder model and default emotion labels")

        return model, emotion_labels

# Load model and labels globally
model, emotion_labels = load_emotion_model()

# -------------------------
# Audio Processing Functions
# -------------------------
def preprocess_audio(signal, sr):
    """Apply comprehensive preprocessing to the audio signal"""
    try:
        # Ensure the sample rate is correct
        if sr != SAMPLE_RATE:
            signal = librosa.resample(signal, orig_sr=sr, target_sr=SAMPLE_RATE)
            sr = SAMPLE_RATE

        # Remove silence from beginning and end
        signal, _ = librosa.effects.trim(signal, top_db=20)

        # Normalize audio to [-1, 1] range
        signal = librosa.util.normalize(signal)

        # Apply noise reduction
        signal = nr.reduce_noise(y=signal, sr=sr, stationary=False)

        # Apply pre-emphasis filter to balance the frequency spectrum
        signal = np.append(signal[0], signal[1:] - 0.97 * signal[:-1])

        return signal, sr
    except Exception as e:
        print(f"Warning in preprocessing: {e}")
        # Return original signal if preprocessing fails
        return librosa.util.normalize(signal), sr

def extract_comprehensive_features(signal, sr):
    """Extract comprehensive audio features for emotion recognition"""
    try:
        # Handle audio length - pad or truncate to fixed length
        if len(signal) > MAX_LEN:
            signal = signal[:MAX_LEN]
        else:
            signal = np.pad(signal, (0, MAX_LEN - len(signal)))

        features = []

        # 1. MFCCs (Mel-frequency cepstral coefficients) - 20 features
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
        mfccs_mean = np.mean(mfccs.T, axis=0)
        mfccs_std = np.std(mfccs.T, axis=0)
        features.extend(mfccs_mean)
        features.extend(mfccs_std)

        # 2. Delta MFCCs (1st order derivatives) - 20 features
        delta_mfccs = librosa.feature.delta(mfccs)
        delta_mfccs_mean = np.mean(delta_mfccs.T, axis=0)
        features.extend(delta_mfccs_mean)

        # 3. Delta-Delta MFCCs (2nd order derivatives) - 20 features
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        delta2_mfccs_mean = np.mean(delta2_mfccs.T, axis=0)
        features.extend(delta2_mfccs_mean)

        # 4. Zero-crossing rate
        zcr = librosa.feature.zero_crossing_rate(y=signal, frame_length=2048, hop_length=512)
        features.append(np.mean(zcr))
        features.append(np.std(zcr))

        # Convert to numpy array and ensure we have exactly 62 features
        features = np.array(features)

        # Pad or truncate to exactly 62 features to match model input
        if len(features) < 62:
            features = np.pad(features, (0, 62 - len(features)))
        elif len(features) > 62:
            features = features[:62]

        print(f"✅ Extracted {len(features)} audio features")
        return features

    except Exception as e:
        print(f"❌ Error in feature extraction: {e}")
        # Return dummy features if extraction fails
        return np.zeros(62)

# -------------------------
# Visualization Functions
# -------------------------
def create_waveform_plot(signal, sr, title="Audio Waveform"):
    """Generate waveform plot"""
    try:
        plt.style.use('dark_background')
        fig, ax = plt.subplots(figsize=(12, 4))

        # Time axis
        time_axis = np.linspace(0, len(signal) / sr, len(signal))

        # Waveform
        ax.plot(time_axis, signal, color='#00d4ff', linewidth=0.8)
        ax.fill_between(time_axis, signal, alpha=0.3, color='#00d4ff')
        ax.set_title(f'{title}', fontsize=14, color='white', pad=15)
        ax.set_xlabel('Time (seconds)', color='white')
        ax.set_ylabel('Amplitude', color='white')
        ax.grid(True, alpha=0.3)
        ax.tick_params(colors='white')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating waveform plot: {e}")
        return create_error_image("Error generating waveform visualization")

def create_mel_spectrogram_plot(signal, sr):
    """Generate Mel Spectrogram visualization"""
    try:
        plt.style.use('dark_background')
        fig, ax = plt.subplots(figsize=(12, 6))

        # Mel Spectrogram
        S = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128,
                                         n_fft=2048, hop_length=512, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)

        img = librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel',
                                      fmax=8000, ax=ax, cmap='viridis')
        ax.set_title('Mel Spectrogram', fontsize=14, color='white', pad=15)
        ax.tick_params(colors='white')
        plt.colorbar(img, ax=ax, format='%+2.0f dB')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating mel spectrogram: {e}")
        return create_error_image("Error generating mel spectrogram")

def create_mfcc_plot(signal, sr):
    """Generate MFCC visualization"""
    try:
        plt.style.use('dark_background')
        fig, ax = plt.subplots(figsize=(12, 6))

        # MFCCs
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
        img = librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax, cmap='coolwarm')
        ax.set_title('MFCCs (Mel-frequency Cepstral Coefficients)', fontsize=14, color='white', pad=15)
        ax.set_ylabel('MFCC Coefficients', color='white')
        ax.set_xlabel('Time (s)', color='white')
        ax.tick_params(colors='white')
        plt.colorbar(img, ax=ax)

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating MFCC plot: {e}")
        return create_error_image("Error generating MFCC visualization")

def create_error_image(error_message):
    """Create an error image when visualization fails"""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, f"❌ {error_message}", transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='red',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.8))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close()
    buf.seek(0)

    return Image.open(buf)

# -------------------------
# Main Prediction Function
# -------------------------
def predict_emotion_comprehensive(audio):
    """Process audio and predict emotion"""

    if audio is None:
        return (
            None,  # audio playback
            create_error_image("No audio provided"),  # waveform
            create_error_image("No audio provided"),  # mel spectrogram
            create_error_image("No audio provided"),  # mfcc
            "❌ No audio provided. Please record or upload audio."  # text result
        )

    try:
        print("🎵 Loading audio file...")
        # Load audio with librosa
        signal, sr = librosa.load(audio, sr=None)
        original_signal = signal.copy()
        original_sr = sr

        print(f"✅ Audio loaded: {len(signal)/sr:.2f}s, {sr}Hz sample rate")

        # Preprocess audio
        print("🔧 Preprocessing audio...")
        processed_signal, sr = preprocess_audio(signal, sr)

        # Generate visualizations
        print("📊 Generating visualizations...")
        waveform_img = create_waveform_plot(original_signal, original_sr, "Original Waveform")
        mel_img = create_mel_spectrogram_plot(processed_signal, sr)
        mfcc_img = create_mfcc_plot(processed_signal, sr)

        # Extract features
        print("🔍 Extracting audio features...")
        features = extract_comprehensive_features(processed_signal, sr)

        # Reshape for model input
        features_reshaped = features.reshape(1, features.shape[0], 1)

        # Make prediction
        print("🧠 Running emotion prediction...")
        prediction = model.predict(features_reshaped, verbose=0)[0]

        # Get the predicted emotion
        predicted_emotion = emotion_labels[np.argmax(prediction)]
        confidence = np.max(prediction) * 100

        # Create result text
        result_lines = [
            f"🎭 **Predicted Emotion: {predicted_emotion.upper()}**",
            f"🎯 **Confidence: {confidence:.1f}%**",
            f"⏱️ **Audio Duration: {len(original_signal)/original_sr:.2f} seconds**",
            f"🔊 **Sample Rate: {original_sr} Hz**",
            "",
            "**Top 3 Predictions:**"
        ]

        # Add top 3 predictions
        top_3_indices = np.argsort(prediction)[::-1][:3]
        for i, idx in enumerate(top_3_indices, 1):
            emotion = emotion_labels[idx]
            prob = prediction[idx] * 100
            result_lines.append(f"{i}. {emotion.capitalize()}: {prob:.1f}%")

        result_text = "\n".join(result_lines)

        print("✅ Analysis complete!")

        return (
            audio,           # audio playback
            waveform_img,    # waveform plot
            mel_img,         # mel spectrogram
            mfcc_img,        # mfcc plot
            result_text      # text result
        )

    except Exception as e:
        error_msg = f"❌ Error processing audio: {str(e)}"
        print(error_msg)

        return (
            None,
            create_error_image("Processing failed"),
            create_error_image("Processing failed"),
            create_error_image("Processing failed"),
            error_msg
        )

# -------------------------
# Gradio Interface
# -------------------------
def create_interface():
    """Create the Gradio interface"""

    custom_css = """
    .gradio-container {
        max-width: 1200px !important;
    }
    .main-header {
        text-align: center;
        background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-size: 2.5em;
        font-weight: bold;
        margin-bottom: 0.5em;
    }
    """

    with gr.Blocks(
        title="🎭 Speech Emotion Recognition",
        theme=gr.themes.Soft(),
        css=custom_css
    ) as interface:

        gr.HTML("""
        <div class="main-header">🎭 Speech Emotion Recognition</div>
        <div style="text-align: center; font-size: 1.1em; color: #666; margin-bottom: 2em;">
            Upload or record audio to analyze emotional content using deep learning
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=1):
                # Audio input
                gr.Markdown("### 🎤 Audio Input")
                audio_input = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Record or Upload Speech"
                )

                analyze_btn = gr.Button(
                    "🚀 Analyze Emotion",
                    variant="primary",
                    size="lg"
                )

                # Audio playback
                gr.Markdown("### 🔊 Audio Playback")
                audio_output = gr.Audio(label="Processed Audio")

            with gr.Column(scale=2):
                # Results
                gr.Markdown("### 📊 Analysis Results")
                result_text = gr.Markdown("Ready for analysis. Upload or record audio and click 'Analyze Emotion'.")

        # Visualizations
        gr.Markdown("### 📈 Visualizations")
        with gr.Tabs():
            with gr.TabItem("🌊 Waveform"):
                waveform_plot = gr.Image(label="Audio Waveform", type="pil")

            with gr.TabItem("🎼 Mel Spectrogram"):
                mel_plot = gr.Image(label="Mel Spectrogram", type="pil")

            with gr.TabItem("📊 MFCC"):
                mfcc_plot = gr.Image(label="MFCC Features", type="pil")

        # Connect button to function
        analyze_btn.click(
            fn=predict_emotion_comprehensive,
            inputs=[audio_input],
            outputs=[audio_output, waveform_plot, mel_plot, mfcc_plot, result_text],
            show_progress=True
        )

        # Auto-trigger on audio input
        audio_input.change(
            fn=predict_emotion_comprehensive,
            inputs=[audio_input],
            outputs=[audio_output, waveform_plot, mel_plot, mfcc_plot, result_text],
            show_progress=True
        )

    return interface

# -------------------------
# Launch
# -------------------------
if __name__ == "__main__":
    print("🚀 Starting Speech Emotion Recognition System...")
    print(f"📊 Model loaded with {len(emotion_labels)} emotion classes: {list(emotion_labels)}")

    interface = create_interface()
    interface.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        debug=True,
        show_error=True
    )

✅ Model loaded successfully!
⚠️ Using default emotion labels
🚀 Starting Speech Emotion Recognition System...
📊 Model loaded with 8 emotion classes: [np.str_('neutral'), np.str_('calm'), np.str_('happy'), np.str_('sad'), np.str_('angry'), np.str_('fearful'), np.str_('disgust'), np.str_('surprised')]
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5282ebd6faef057a05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎵 Loading audio file...
✅ Audio loaded: 3.30s, 48000Hz sample rate
🔧 Preprocessing audio...
📊 Generating visualizations...
🔍 Extracting audio features...
✅ Extracted 62 audio features
🧠 Running emotion prediction...
✅ Analysis complete!
🎵 Loading audio file...
✅ Audio loaded: 3.30s, 48000Hz sample rate
🔧 Preprocessing audio...
📊 Generating visualizations...
🔍 Extracting audio features...
✅ Extracted 62 audio features
🧠 Running emotion prediction...
✅ Analysis complete!
🎵 Loading audio file...
✅ Audio loaded: 3.30s, 48000Hz sample rate
🔧 Preprocessing audio...
📊 Generating visualizations...
🔍 Extracting audio features...
✅ Extracted 62 audio features
🧠 Running emotion prediction...
✅ Analysis complete!


In [ ]:
# import gradio as gr
# import librosa
# import librosa.display
# import numpy as np
# import matplotlib.pyplot as plt
# import tensorflow as tf
# import noisereduce as nr
# from tensorflow.keras.models import load_model
# import io
# import os
# import time

# # -------------------------
# # Configuration
# # -------------------------
# SAMPLE_RATE = 22050
# MAX_DURATION = 3  # seconds
# MAX_LEN = SAMPLE_RATE * MAX_DURATION

# # Install required packages
# !pip install gradio librosa matplotlib tensorflow noisereduce

# # -------------------------
# # Load model and emotion classes
# # -------------------------
# try:
#     model = load_model("/content/emotion_recognition_model.h5")
#     print("✅ Model loaded successfully!")

#     # Try to load the classes from the saved file
#     if os.path.exists('emotion_classes.npy'):
#         emotion_labels = np.load('emotion_classes.npy', allow_pickle=True)
#         print(f"✅ Emotion classes loaded: {emotion_labels}")
#     else:
#         # Default emotion labels if file not found
#         emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
#         print("⚠️ Using default emotion labels")
# except Exception as e:
#     print(f"❌ Error loading model: {e}")
#     print("⚠️ Using a placeholder model for demonstration")
#     # Create a dummy model for demonstration purposes
#     from tensorflow.keras.layers import Input, Dense

#     input_shape = (62, 1)  # Example shape for features
#     inputs = Input(shape=input_shape)
#     x = Dense(64, activation='relu')(inputs)
#     outputs = Dense(8, activation='softmax')(x)
#     model = tf.keras.Model(inputs=inputs, outputs=outputs)

#     # Default emotion labels
#     emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
#     print("⚠️ Using default emotion labels for demonstration")

# # -------------------------
# # Audio Processing Functions
# # -------------------------
# def preprocess_audio(signal, sr):
#     """Apply preprocessing to the audio signal"""
#     # Ensure the sample rate is correct
#     if sr != SAMPLE_RATE:
#         signal = librosa.resample(signal, orig_sr=sr, target_sr=SAMPLE_RATE)
#         sr = SAMPLE_RATE

#     # Normalize audio
#     signal = librosa.util.normalize(signal)

#     # Denoise
#     signal = nr.reduce_noise(y=signal, sr=sr)

#     return signal, sr

# def extract_features(signal, sr):
#     """Extract audio features for emotion recognition"""
#     # Handle audio length
#     if len(signal) > MAX_LEN:
#         signal = signal[:MAX_LEN]
#     else:
#         signal = np.pad(signal, (0, MAX_LEN - len(signal)))

#     # Extract MFCCs (increased to 20 for better feature representation)
#     mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
#     mfccs_mean = np.mean(mfccs.T, axis=0)

#     # Extract delta MFCCs (1st order derivatives)
#     delta_mfccs = librosa.feature.delta(mfccs_mean)

#     # Zero-crossing rate - voice characteristics
#     zcr = librosa.feature.zero_crossing_rate(y=signal)
#     zcr_mean = np.mean(zcr)

#     # Root mean square energy - volume/intensity
#     rmse = librosa.feature.rms(y=signal)
#     rmse_mean = np.mean(rmse)

#     # Spectral centroid - brightness of sound
#     spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr)
#     spectral_centroid_mean = np.mean(spectral_centroid)

#     # Chroma features - tonal content
#     chroma = librosa.feature.chroma_stft(y=signal, sr=sr)
#     chroma_mean = np.mean(chroma.T, axis=0)

#     # Spectral contrast - difference between peaks and valleys
#     spectral_contrast = librosa.feature.spectral_contrast(y=signal, sr=sr)
#     spectral_contrast_mean = np.mean(spectral_contrast.T, axis=0)

#     # Combine all features
#     features = np.hstack([
#         mfccs_mean,            # 20 features
#         delta_mfccs,           # 20 features
#         zcr_mean,              # 1 feature
#         rmse_mean,             # 1 feature
#         spectral_centroid_mean,# 1 feature
#         chroma_mean,           # 12 features
#         spectral_contrast_mean # 7 features
#     ])

#     return features

# # -------------------------
# # Visualization Functions
# # -------------------------
# def plot_waveform(signal, sr, title="Waveform"):
#     """Generate waveform plot"""
#     plt.figure(figsize=(8, 2))
#     plt.title(title)
#     librosa.display.waveshow(signal, sr=sr)
#     plt.tight_layout()

#     # Save to BytesIO
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', dpi=100)
#     plt.close()
#     buf.seek(0)
#     return buf

# def plot_mel_spectrogram(signal, sr):
#     """Generate Mel Spectrogram visualization"""
#     plt.figure(figsize=(8, 3))
#     S = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128, fmax=8000)
#     S_dB = librosa.power_to_db(S, ref=np.max)
#     librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', fmax=8000)
#     plt.colorbar(format='%+2.0f dB')
#     plt.title('Mel Spectrogram')
#     plt.tight_layout()

#     # Save to BytesIO
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', dpi=100)
#     plt.close()
#     buf.seek(0)
#     return buf

# def plot_mfcc(signal, sr):
#     """Generate MFCC visualization"""
#     plt.figure(figsize=(8, 3))
#     mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
#     librosa.display.specshow(mfccs, sr=sr, x_axis='time')
#     plt.colorbar(format='%+2.0f')
#     plt.title('MFCCs')
#     plt.tight_layout()

#     # Save to BytesIO
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', dpi=100)
#     plt.close()
#     buf.seek(0)
#     return buf

# def plot_prediction_pie(prediction_probs):
#     """Generate pie chart for emotion probabilities"""
#     # Sort probabilities in descending order
#     sorted_indices = np.argsort(prediction_probs)[::-1]
#     sorted_probs = prediction_probs[sorted_indices]
#     sorted_labels = emotion_labels[sorted_indices]

#     # Use only top 4 emotions for cleaner display
#     top_n = 4
#     others_prob = np.sum(sorted_probs[top_n:]) if len(sorted_probs) > top_n else 0
#     top_probs = sorted_probs[:top_n]
#     top_labels = sorted_labels[:top_n]

#     # Add "others" if there are more than top_n emotions
#     if others_prob > 0:
#         top_probs = np.append(top_probs, others_prob)
#         top_labels = np.append(top_labels, 'others')

#     # Generate nice colors
#     colors = plt.cm.viridis(np.linspace(0, 0.8, len(top_probs)))

#     # Create a pie chart
#     plt.figure(figsize=(6, 6))
#     wedges, texts, autotexts = plt.pie(
#         top_probs,
#         labels=top_labels,
#         autopct='%1.1f%%',
#         startangle=90,
#         colors=colors,
#         wedgeprops={'edgecolor': 'white', 'linewidth': 1}
#     )

#     # Style the percentage text
#     for autotext in autotexts:
#         autotext.set_color('white')
#         autotext.set_fontsize(12)
#         autotext.set_fontweight('bold')

#     plt.axis('equal')
#     plt.title("Emotion Prediction Probabilities", fontsize=14, pad=20)
#     plt.tight_layout()

#     # Save to BytesIO
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', dpi=100)
#     plt.close()
#     buf.seek(0)
#     return buf

# # -------------------------
# # Main Prediction Function
# # -------------------------
# def predict_emotion(audio):
#     """Process audio and predict emotion"""
#     # Generate a unique timestamp for saving files
#     timestamp = int(time.time())

#     # If no audio is provided
#     if audio is None:
#         return (
#             None,  # audio playback
#             None,  # waveform
#             None,  # mel spectrogram
#             None,  # mfcc
#             None,  # prediction pie
#             "No audio provided. Please record or upload audio."  # text result
#         )

#     try:
#         # Load audio
#         signal, sr = librosa.load(audio, sr=None)  # Keep original sample rate initially

#         # Create a copy of the original signal for visualization
#         original_signal = signal.copy()

#         # Generate plots for the original audio
#         waveform_img = plot_waveform(original_signal, sr, "Original Waveform")

#         # Preprocess audio
#         processed_signal, sr = preprocess_audio(signal, sr)

#         # Generate plots for the processed audio
#         mel_img = plot_mel_spectrogram(processed_signal, sr)
#         mfcc_img = plot_mfcc(processed_signal, sr)

#         # Extract features
#         features = extract_features(processed_signal, sr)

#         # Reshape for model input (samples, timesteps, features)
#         features = features.reshape(1, features.shape[0], 1)

#         # Make prediction
#         prediction = model.predict(features)[0]

#         # Get the predicted emotion
#         predicted_emotion = emotion_labels[np.argmax(prediction)]
#         confidence = np.max(prediction) * 100

#         # Generate prediction pie chart
#         pie_img = plot_prediction_pie(prediction)

#         # Format the result text
#         result = f"Predicted Emotion: {predicted_emotion} (Confidence: {confidence:.1f}%)"

#         return (
#             audio,         # audio playback
#             waveform_img,  # waveform plot
#             mel_img,       # mel spectrogram
#             mfcc_img,      # mfcc plot
#             pie_img,       # prediction pie chart
#             result         # text result
#         )

#     except Exception as e:
#         print(f"Error processing audio: {e}")
#         return (
#             None,  # audio playback
#             None,  # waveform
#             None,  # mel spectrogram
#             None,  # mfcc
#             None,  # prediction pie
#             f"Error processing audio: {str(e)}"  # text result
#         )

# # -------------------------
# # Gradio Interface
# # -------------------------
# with gr.Blocks(title="🎭 Speech Emotion Recognition", theme=gr.themes.Soft()) as interface:
#     gr.Markdown("""
#     # 🎭 Speech Emotion Recognition

#     Record or upload a short audio clip (ideally 3 seconds) to analyze its emotional content.
#     The model will predict one of eight emotions: neutral, calm, happy, sad, angry, fearful, disgust, or surprised.

#     **Instructions:**
#     1. Record audio using your microphone or upload an audio file
#     2. Click "Submit" to analyze the emotion
#     3. View the visualizations and prediction results
#     """)

#     with gr.Row():
#         with gr.Column(scale=1):
#             # Input components
#             audio_input = gr.Audio(
#                 sources=["microphone", "upload"],
#                 type="filepath",
#                 label="🎤 Record or Upload Speech (3 seconds recommended)"
#             )
#             submit_btn = gr.Button("🔍 Analyze Emotion", variant="primary")

#         with gr.Column(scale=2):
#             # Results text
#             result_text = gr.Textbox(label="🎯 Prediction Result")

#             # Visualization outputs
#             with gr.Tabs():
#                 with gr.TabItem("Audio Waveform"):
#                     waveform_output = gr.Image(label="Audio Waveform")

#                 with gr.TabItem("Mel Spectrogram"):
#                     spectrogram_output = gr.Image(label="Mel Spectrogram")

#                 with gr.TabItem("MFCCs"):
#                     mfcc_output = gr.Image(label="MFCC Features")

#                 with gr.TabItem("Prediction"):
#                     pie_output = gr.Image(label="Emotion Probabilities")

#             # Audio playback
#             audio_output = gr.Audio(label="🔊 Audio Playback")

#     # Set up the submit action
#     submit_btn.click(
#         fn=predict_emotion,
#         inputs=[audio_input],
#         outputs=[audio_output, waveform_output, spectrogram_output, mfcc_output, pie_output, result_text]
#     )

#     gr.Markdown("""
#     ## About this application

#     This application uses a deep learning model to recognize emotions in speech.
#     The model analyzes various audio features such as:

#     - Mel-frequency cepstral coefficients (MFCCs)
#     - Zero-crossing rate
#     - Root mean square energy
#     - Spectral centroid
#     - Chroma features

#     The model architecture combines convolutional neural networks (CNNs) and
#     long short-term memory networks (LSTMs) for robust emotion recognition.
#     """)

# # Launch the interface with queue enabled for better handling of concurrent users
# interface.launch(share=True, debug=True, enable_queue=True)

✅ Model loaded successfully!
✅ Emotion classes loaded: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


TypeError: Blocks.launch() got an unexpected keyword argument 'enable_queue'

In [ ]:
# import gradio as gr
# import librosa
# import librosa.display
# import numpy as np
# import matplotlib.pyplot as plt
# import tensorflow as tf
# import noisereduce as nr
# from tensorflow.keras.models import load_model
# import io
# import os
# import time

# # -------------------------
# # Configuration
# # -------------------------
# SAMPLE_RATE = 22050
# MAX_DURATION = 3  # seconds
# MAX_LEN = SAMPLE_RATE * MAX_DURATION

# # Install required packages
# !pip install gradio librosa matplotlib tensorflow noisereduce

# # -------------------------
# # Load model and emotion classes
# # -------------------------
# try:
#     model = load_model("/content/emotion_recognition_model.h5")
#     print("✅ Model loaded successfully!")

#     # Try to load the classes from the saved file
#     if os.path.exists('emotion_classes.npy'):
#         emotion_labels = np.load('emotion_classes.npy', allow_pickle=True)
#         print(f"✅ Emotion classes loaded: {emotion_labels}")
#     else:
#         # Default emotion labels if file not found
#         emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
#         print("⚠️ Using default emotion labels")
# except Exception as e:
#     print(f"❌ Error loading model: {e}")
#     print("⚠️ Using a placeholder model for demonstration")
#     # Create a dummy model for demonstration purposes
#     from tensorflow.keras.layers import Input, Dense

#     input_shape = (62, 1)  # Example shape for features
#     inputs = Input(shape=input_shape)
#     x = Dense(64, activation='relu')(inputs)
#     outputs = Dense(8, activation='softmax')(x)
#     model = tf.keras.Model(inputs=inputs, outputs=outputs)

#     # Default emotion labels
#     emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
#     print("⚠️ Using default emotion labels for demonstration")

# # -------------------------
# # Audio Processing Functions
# # -------------------------
# def preprocess_audio(signal, sr):
#     """Apply preprocessing to the audio signal"""
#     # Ensure the sample rate is correct
#     if sr != SAMPLE_RATE:
#         signal = librosa.resample(signal, orig_sr=sr, target_sr=SAMPLE_RATE)
#         sr = SAMPLE_RATE

#     # Normalize audio
#     signal = librosa.util.normalize(signal)

#     # Denoise
#     signal = nr.reduce_noise(y=signal, sr=sr)

#     return signal, sr

# def extract_features(signal, sr):
#     """Extract audio features for emotion recognition"""
#     # Handle audio length
#     if len(signal) > MAX_LEN:
#         signal = signal[:MAX_LEN]
#     else:
#         signal = np.pad(signal, (0, MAX_LEN - len(signal)))

#     # Extract MFCCs (increased to 20 for better feature representation)
#     mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
#     mfccs_mean = np.mean(mfccs.T, axis=0)

#     # Extract delta MFCCs (1st order derivatives)
#     delta_mfccs = librosa.feature.delta(mfccs_mean)

#     # Zero-crossing rate - voice characteristics
#     zcr = librosa.feature.zero_crossing_rate(y=signal)
#     zcr_mean = np.mean(zcr)

#     # Root mean square energy - volume/intensity
#     rmse = librosa.feature.rms(y=signal)
#     rmse_mean = np.mean(rmse)

#     # Spectral centroid - brightness of sound
#     spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr)
#     spectral_centroid_mean = np.mean(spectral_centroid)

#     # Chroma features - tonal content
#     chroma = librosa.feature.chroma_stft(y=signal, sr=sr)
#     chroma_mean = np.mean(chroma.T, axis=0)

#     # Spectral contrast - difference between peaks and valleys
#     spectral_contrast = librosa.feature.spectral_contrast(y=signal, sr=sr)
#     spectral_contrast_mean = np.mean(spectral_contrast.T, axis=0)

#     # Combine all features
#     features = np.hstack([
#         mfccs_mean,            # 20 features
#         delta_mfccs,           # 20 features
#         zcr_mean,              # 1 feature
#         rmse_mean,             # 1 feature
#         spectral_centroid_mean,# 1 feature
#         chroma_mean,           # 12 features
#         spectral_contrast_mean # 7 features
#     ])

#     return features

# # -------------------------
# # Visualization Functions
# # -------------------------
# def plot_waveform(signal, sr, title="Waveform"):
#     """Generate waveform plot"""
#     plt.figure(figsize=(8, 2))
#     plt.title(title)
#     librosa.display.waveshow(signal, sr=sr)
#     plt.tight_layout()

#     # Save to BytesIO
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', dpi=100)
#     plt.close()
#     buf.seek(0)
#     return buf

# def plot_mel_spectrogram(signal, sr):
#     """Generate Mel Spectrogram visualization"""
#     plt.figure(figsize=(8, 3))
#     S = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128, fmax=8000)
#     S_dB = librosa.power_to_db(S, ref=np.max)
#     librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', fmax=8000)
#     plt.colorbar(format='%+2.0f dB')
#     plt.title('Mel Spectrogram')
#     plt.tight_layout()

#     # Save to BytesIO
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', dpi=100)
#     plt.close()
#     buf.seek(0)
#     return buf

# def plot_mfcc(signal, sr):
#     """Generate MFCC visualization"""
#     plt.figure(figsize=(8, 3))
#     mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
#     librosa.display.specshow(mfccs, sr=sr, x_axis='time')
#     plt.colorbar(format='%+2.0f')
#     plt.title('MFCCs')
#     plt.tight_layout()

#     # Save to BytesIO
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', dpi=100)
#     plt.close()
#     buf.seek(0)
#     return buf

# def plot_prediction_pie(prediction_probs):
#     """Generate pie chart for emotion probabilities"""
#     # Sort probabilities in descending order
#     sorted_indices = np.argsort(prediction_probs)[::-1]
#     sorted_probs = prediction_probs[sorted_indices]
#     sorted_labels = emotion_labels[sorted_indices]

#     # Use only top 4 emotions for cleaner display
#     top_n = 4
#     others_prob = np.sum(sorted_probs[top_n:]) if len(sorted_probs) > top_n else 0
#     top_probs = sorted_probs[:top_n]
#     top_labels = sorted_labels[:top_n]

#     # Add "others" if there are more than top_n emotions
#     if others_prob > 0:
#         top_probs = np.append(top_probs, others_prob)
#         top_labels = np.append(top_labels, 'others')

#     # Generate nice colors
#     colors = plt.cm.viridis(np.linspace(0, 0.8, len(top_probs)))

#     # Create a pie chart
#     plt.figure(figsize=(6, 6))
#     wedges, texts, autotexts = plt.pie(
#         top_probs,
#         labels=top_labels,
#         autopct='%1.1f%%',
#         startangle=90,
#         colors=colors,
#         wedgeprops={'edgecolor': 'white', 'linewidth': 1}
#     )

#     # Style the percentage text
#     for autotext in autotexts:
#         autotext.set_color('white')
#         autotext.set_fontsize(12)
#         autotext.set_fontweight('bold')

#     plt.axis('equal')
#     plt.title("Emotion Prediction Probabilities", fontsize=14, pad=20)
#     plt.tight_layout()

#     # Save to BytesIO
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', dpi=100)
#     plt.close()
#     buf.seek(0)
#     return buf

# # -------------------------
# # Main Prediction Function
# # -------------------------
# def predict_emotion(audio):
#     """Process audio and predict emotion"""
#     # Generate a unique timestamp for saving files
#     timestamp = int(time.time())

#     # If no audio is provided
#     if audio is None:
#         return (
#             None,  # audio playback
#             None,  # waveform
#             None,  # mel spectrogram
#             None,  # mfcc
#             None,  # prediction pie
#             "No audio provided. Please record or upload audio."  # text result
#         )

#     try:
#         # Load audio
#         signal, sr = librosa.load(audio, sr=None)  # Keep original sample rate initially

#         # Create a copy of the original signal for visualization
#         original_signal = signal.copy()

#         # Generate plots for the original audio
#         waveform_img = plot_waveform(original_signal, sr, "Original Waveform")

#         # Preprocess audio
#         processed_signal, sr = preprocess_audio(signal, sr)

#         # Generate plots for the processed audio
#         mel_img = plot_mel_spectrogram(processed_signal, sr)
#         mfcc_img = plot_mfcc(processed_signal, sr)

#         # Extract features
#         features = extract_features(processed_signal, sr)

#         # Reshape for model input (samples, timesteps, features)
#         features = features.reshape(1, features.shape[0], 1)

#         # Make prediction
#         prediction = model.predict(features)[0]

#         # Get the predicted emotion
#         predicted_emotion = emotion_labels[np.argmax(prediction)]
#         confidence = np.max(prediction) * 100

#         # Generate prediction pie chart
#         pie_img = plot_prediction_pie(prediction)

#         # Format the result text
#         result = f"Predicted Emotion: {predicted_emotion} (Confidence: {confidence:.1f}%)"

#         return (
#             audio,         # audio playback
#             waveform_img,  # waveform plot
#             mel_img,       # mel spectrogram
#             mfcc_img,      # mfcc plot
#             pie_img,       # prediction pie chart
#             result         # text result
#         )

#     except Exception as e:
#         print(f"Error processing audio: {e}")
#         return (
#             None,  # audio playback
#             None,  # waveform
#             None,  # mel spectrogram
#             None,  # mfcc
#             None,  # prediction pie
#             f"Error processing audio: {str(e)}"  # text result
#         )

# # -------------------------
# # Gradio Interface
# # -------------------------
# with gr.Blocks(title="🎭 Speech Emotion Recognition", theme=gr.themes.Soft()) as interface:
#     gr.Markdown("""
#     # 🎭 Speech Emotion Recognition

#     Record or upload a short audio clip (ideally 3 seconds) to analyze its emotional content.
#     The model will predict one of eight emotions: neutral, calm, happy, sad, angry, fearful, disgust, or surprised.

#     **Instructions:**
#     1. Record audio using your microphone or upload an audio file
#     2. Click "Submit" to analyze the emotion
#     3. View the visualizations and prediction results
#     """)

#     with gr.Row():
#         with gr.Column(scale=1):
#             # Input components
#             audio_input = gr.Audio(
#                 sources=["microphone", "upload"],
#                 type="filepath",
#                 label="🎤 Record or Upload Speech (3 seconds recommended)"
#             )
#             submit_btn = gr.Button("🔍 Analyze Emotion", variant="primary")

#         with gr.Column(scale=2):
#             # Results text
#             result_text = gr.Textbox(label="🎯 Prediction Result")

#             # Visualization outputs
#             with gr.Tabs():
#                 with gr.TabItem("Audio Waveform"):
#                     waveform_output = gr.Image(label="Audio Waveform")

#                 with gr.TabItem("Mel Spectrogram"):
#                     spectrogram_output = gr.Image(label="Mel Spectrogram")

#                 with gr.TabItem("MFCCs"):
#                     mfcc_output = gr.Image(label="MFCC Features")

#                 with gr.TabItem("Prediction"):
#                     pie_output = gr.Image(label="Emotion Probabilities")

#             # Audio playback
#             audio_output = gr.Audio(label="🔊 Audio Playback")

#     # Set up the submit action
#     submit_btn.click(
#         fn=predict_emotion,
#         inputs=[audio_input],
#         outputs=[audio_output, waveform_output, spectrogram_output, mfcc_output, pie_output, result_text]
#     )

#     gr.Markdown("""
#     ## About this application

#     This application uses a deep learning model to recognize emotions in speech.
#     The model analyzes various audio features such as:

#     - Mel-frequency cepstral coefficients (MFCCs)
#     - Zero-crossing rate
#     - Root mean square energy
#     - Spectral centroid
#     - Chroma features

#     The model architecture combines convolutional neural networks (CNNs) and
#     long short-term memory networks (LSTMs) for robust emotion recognition.
#     """)

# # Launch the interface with queue enabled for better handling of concurrent users
# interface.launch(share=True, debug=True)

ERROR: Operation cancelled by user


✅ Model loaded successfully!
✅ Emotion classes loaded: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e3aa43ed006d6eaada.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0dbf741ae55417db33.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://e3aa43ed006d6eaada.gradio.live


In [ ]:
import gradio as gr
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import noisereduce as nr
from tensorflow.keras.models import load_model
from PIL import Image
import io
import os
import time
import warnings
warnings.filterwarnings('ignore')

# -------------------------
# Configuration
# -------------------------
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Install required packages (uncomment if running in Colab)
# !pip install gradio librosa matplotlib tensorflow noisereduce

# -------------------------
# Load model and emotion classes
# -------------------------
def load_emotion_model():
    """Load the emotion recognition model with proper error handling"""
    try:
        model = load_model("/content/emotion_recognition_model.h5")
        print("✅ Model loaded successfully!")

        # Try to load the classes from the saved file
        if os.path.exists('emotion_classes.npy'):
            emotion_labels = np.load('emotion_classes.npy', allow_pickle=True)
            print(f"✅ Emotion classes loaded: {emotion_labels}")
        else:
            # Default emotion labels if file not found
            emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
            print("⚠️ Using default emotion labels")

        return model, emotion_labels

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("⚠️ Creating a placeholder model for demonstration")

        # Create a dummy model for demonstration purposes
        from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
        from tensorflow.keras.models import Model

        # More realistic model architecture
        inputs = Input(shape=(62, 1))
        x = LSTM(128, return_sequences=True, dropout=0.2)(inputs)
        x = LSTM(64, dropout=0.2)(x)
        x = Dense(32, activation='relu')(x)
        x = Dropout(0.3)(x)
        outputs = Dense(8, activation='softmax')(x)

        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

        # Default emotion labels
        emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
        print("⚠️ Using placeholder model and default emotion labels")

        return model, emotion_labels

# Load model and labels globally
model, emotion_labels = load_emotion_model()

# -------------------------
# Audio Processing Functions
# -------------------------
def preprocess_audio(signal, sr):
    """Apply comprehensive preprocessing to the audio signal"""
    try:
        # Ensure the sample rate is correct
        if sr != SAMPLE_RATE:
            signal = librosa.resample(signal, orig_sr=sr, target_sr=SAMPLE_RATE)
            sr = SAMPLE_RATE

        # Remove silence from beginning and end
        signal, _ = librosa.effects.trim(signal, top_db=20)

        # Normalize audio to [-1, 1] range
        signal = librosa.util.normalize(signal)

        # Apply noise reduction
        signal = nr.reduce_noise(y=signal, sr=sr, stationary=False)

        # Apply pre-emphasis filter to balance the frequency spectrum
        signal = np.append(signal[0], signal[1:] - 0.97 * signal[:-1])

        return signal, sr
    except Exception as e:
        print(f"Warning in preprocessing: {e}")
        # Return original signal if preprocessing fails
        return librosa.util.normalize(signal), sr

def extract_comprehensive_features(signal, sr):
    """Extract comprehensive audio features for emotion recognition"""
    try:
        # Handle audio length - pad or truncate to fixed length
        if len(signal) > MAX_LEN:
            signal = signal[:MAX_LEN]
        else:
            signal = np.pad(signal, (0, MAX_LEN - len(signal)))

        features = []

        # 1. MFCCs (Mel-frequency cepstral coefficients) - 20 features
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
        mfccs_mean = np.mean(mfccs.T, axis=0)
        mfccs_std = np.std(mfccs.T, axis=0)
        features.extend(mfccs_mean)
        features.extend(mfccs_std)  # Add standard deviation for more robustness

        # 2. Delta MFCCs (1st order derivatives) - 20 features
        delta_mfccs = librosa.feature.delta(mfccs)
        delta_mfccs_mean = np.mean(delta_mfccs.T, axis=0)
        features.extend(delta_mfccs_mean)

        # 3. Delta-Delta MFCCs (2nd order derivatives) - 20 features
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        delta2_mfccs_mean = np.mean(delta2_mfccs.T, axis=0)
        features.extend(delta2_mfccs_mean)

        # 4. Zero-crossing rate - voice characteristics
        zcr = librosa.feature.zero_crossing_rate(y=signal, frame_length=2048, hop_length=512)
        features.append(np.mean(zcr))
        features.append(np.std(zcr))

        # 5. Root mean square energy - volume/intensity
        rmse = librosa.feature.rms(y=signal, frame_length=2048, hop_length=512)
        features.append(np.mean(rmse))
        features.append(np.std(rmse))

        # 6. Spectral centroid - brightness of sound
        spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_centroid))
        features.append(np.std(spectral_centroid))

        # 7. Spectral rolloff - frequency below which 85% of energy is contained
        spectral_rolloff = librosa.feature.spectral_rolloff(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_rolloff))
        features.append(np.std(spectral_rolloff))

        # 8. Spectral bandwidth - width of the frequency band
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_bandwidth))
        features.append(np.std(spectral_bandwidth))

        # Convert to numpy array and ensure we have exactly 62 features
        features = np.array(features)

        # Pad or truncate to exactly 62 features to match model input
        if len(features) < 62:
            features = np.pad(features, (0, 62 - len(features)))
        elif len(features) > 62:
            features = features[:62]

        print(f"✅ Extracted {len(features)} audio features")
        return features

    except Exception as e:
        print(f"❌ Error in feature extraction: {e}")
        # Return dummy features if extraction fails
        return np.zeros(62)  # Match expected feature count

# -------------------------
# Enhanced Visualization Functions
# -------------------------
def create_waveform_plot(signal, sr, title="Audio Waveform"):
    """Generate enhanced waveform plot and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

        # Time axis
        time_axis = np.linspace(0, len(signal) / sr, len(signal))

        # Original waveform
        ax1.plot(time_axis, signal, color='#00d4ff', linewidth=0.8, alpha=0.8)
        ax1.fill_between(time_axis, signal, alpha=0.3, color='#00d4ff')
        ax1.set_title(f'{title} - Time Domain', fontsize=12, color='white', pad=10)
        ax1.set_xlabel('Time (seconds)', color='white')
        ax1.set_ylabel('Amplitude', color='white')
        ax1.grid(True, alpha=0.3)
        ax1.tick_params(colors='white')

        # Spectrogram
        D = librosa.amplitude_to_db(np.abs(librosa.stft(signal)), ref=np.max)
        img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz',
                                      ax=ax2, cmap='plasma', alpha=0.8)
        ax2.set_title('Spectrogram - Frequency Domain', fontsize=12, color='white', pad=10)
        ax2.tick_params(colors='white')
        plt.colorbar(img, ax=ax2, format='%+2.0f dB')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating waveform plot: {e}")
        return create_error_image("Error generating waveform visualization")

def create_mel_spectrogram_plot(signal, sr):
    """Generate enhanced Mel Spectrogram visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # Mel Spectrogram
        S = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128,
                                         n_fft=2048, hop_length=512, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)

        img1 = librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel',
                                       fmax=8000, ax=ax1, cmap='viridis')
        ax1.set_title('Mel Spectrogram', fontsize=14, color='white', pad=15)
        ax1.tick_params(colors='white')
        plt.colorbar(img1, ax=ax1, format='%+2.0f dB')

        # Mel filter bank visualization
        mel_filters = librosa.filters.mel(sr=sr, n_fft=2048, n_mels=10, fmax=8000)
        ax2.plot(mel_filters.T, alpha=0.8, linewidth=1.5)
        ax2.set_title('Mel Filter Bank (First 10 filters)', fontsize=14, color='white', pad=15)
        ax2.set_xlabel('Frequency Bins', color='white')
        ax2.set_ylabel('Filter Response', color='white')
        ax2.grid(True, alpha=0.3)
        ax2.tick_params(colors='white')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating mel spectrogram: {e}")
        return create_error_image("Error generating mel spectrogram")

def create_mfcc_plot(signal, sr):
    """Generate enhanced MFCC visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

        # MFCCs
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
        img1 = librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax1, cmap='coolwarm')
        ax1.set_title('MFCCs (Mel-frequency Cepstral Coefficients)', fontsize=12, color='white')
        ax1.set_ylabel('MFCC Coefficients', color='white')
        ax1.tick_params(colors='white')
        plt.colorbar(img1, ax=ax1)

        # Delta MFCCs
        delta_mfccs = librosa.feature.delta(mfccs)
        img2 = librosa.display.specshow(delta_mfccs, sr=sr, x_axis='time', ax=ax2, cmap='coolwarm')
        ax2.set_title('Delta MFCCs (1st derivatives)', fontsize=12, color='white')
        ax2.set_ylabel('Delta MFCC', color='white')
        ax2.tick_params(colors='white')
        plt.colorbar(img2, ax=ax2)

        # Delta-Delta MFCCs
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        img3 = librosa.display.specshow(delta2_mfccs, sr=sr, x_axis='time', ax=ax3, cmap='coolwarm')
        ax3.set_title('Delta-Delta MFCCs (2nd derivatives)', fontsize=12, color='white')
        ax3.set_ylabel('Delta-Delta MFCC', color='white')
        ax3.set_xlabel('Time (s)', color='white')
        ax3.tick_params(colors='white')
        plt.colorbar(img3, ax=ax3)

        # MFCC statistics
        mfcc_mean = np.mean(mfccs.T, axis=0)
        mfcc_std = np.std(mfccs.T, axis=0)

        x_pos = np.arange(len(mfcc_mean))
        ax4.bar(x_pos - 0.2, mfcc_mean, 0.4, label='Mean', alpha=0.8, color='#ff6b6b')
        ax4.bar(x_pos + 0.2, mfcc_std, 0.4, label='Std Dev', alpha=0.8, color='#4ecdc4')
        ax4.set_title('MFCC Statistics', fontsize=12, color='white')
        ax4.set_xlabel('MFCC Coefficient Index', color='white')
        ax4.set_ylabel('Value', color='white')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        ax4.tick_params(colors='white')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating MFCC plot: {e}")
        return create_error_image("Error generating MFCC visualization")

def create_prediction_plot(prediction_probs, feature_importance=None):
    """Generate comprehensive prediction visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

        # 1. Horizontal bar chart of predictions
        sorted_indices = np.argsort(prediction_probs)
        sorted_probs = prediction_probs[sorted_indices]
        sorted_labels = emotion_labels[sorted_indices]

        colors = plt.cm.viridis(np.linspace(0, 1, len(sorted_probs)))
        bars = ax1.barh(sorted_labels, sorted_probs * 100, color=colors, alpha=0.8)
        ax1.set_xlabel('Probability (%)', color='white', fontsize=12)
        ax1.set_title('Emotion Prediction Probabilities', color='white', fontsize=14, pad=15)
        ax1.tick_params(colors='white')
        ax1.grid(True, alpha=0.3, axis='x')

        # Add percentage labels on bars
        for bar, prob in zip(bars, sorted_probs):
            width = bar.get_width()
            ax1.text(width + 1, bar.get_y() + bar.get_height()/2,
                    f'{prob*100:.1f}%', ha='left', va='center', color='white', fontsize=10)

        # 2. Pie chart for top emotions
        top_n = 5
        top_indices = np.argsort(prediction_probs)[::-1][:top_n]
        top_probs = prediction_probs[top_indices]
        top_labels = emotion_labels[top_indices]

        # Add "others" category if needed
        others_prob = 1 - np.sum(top_probs)
        if others_prob > 0.01:  # Only add if significant
            top_probs = np.append(top_probs, others_prob)
            top_labels = np.append(top_labels, 'Others')

        colors_pie = plt.cm.Set3(np.linspace(0, 1, len(top_probs)))
        wedges, texts, autotexts = ax2.pie(top_probs, labels=top_labels, autopct='%1.1f%%',
                                          startangle=90, colors=colors_pie,
                                          textprops={'color': 'white', 'fontsize': 10})
        ax2.set_title('Top Emotion Predictions', color='white', fontsize=14, pad=15)

        # 3. Confidence analysis
        max_prob = np.max(prediction_probs)
        entropy = -np.sum(prediction_probs * np.log2(prediction_probs + 1e-10))
        max_entropy = np.log2(len(emotion_labels))
        normalized_entropy = entropy / max_entropy

        confidence_metrics = ['Max Probability', 'Certainty\n(1 - Entropy)', 'Top-2 Gap']
        top2_gap = sorted_probs[-1] - sorted_probs[-2] if len(sorted_probs) > 1 else max_prob
        confidence_values = [max_prob, 1 - normalized_entropy, top2_gap]

        bars3 = ax3.bar(confidence_metrics, confidence_values,
                       color=['#ff6b6b', '#4ecdc4', '#45b7d1'], alpha=0.8)
        ax3.set_title('Prediction Confidence Metrics', color='white', fontsize=14, pad=15)
        ax3.set_ylabel('Score', color='white')
        ax3.tick_params(colors='white')
        ax3.grid(True, alpha=0.3, axis='y')

        # Add value labels on bars
        for bar, value in zip(bars3, confidence_values):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{value:.3f}', ha='center', va='bottom', color='white', fontsize=11)

        # 4. Prediction distribution
        ax4.plot(emotion_labels, prediction_probs * 100, 'o-',
                color='#ffa726', linewidth=2, markersize=8, alpha=0.8)
        ax4.fill_between(range(len(emotion_labels)), prediction_probs * 100,
                        alpha=0.3, color='#ffa726')
        ax4.set_title('Prediction Distribution', color='white', fontsize=14, pad=15)
        ax4.set_ylabel('Probability (%)', color='white')
        ax4.set_xlabel('Emotions', color='white')
        ax4.tick_params(colors='white', axis='x', rotation=45)
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating prediction plot: {e}")
        return create_error_image("Error generating prediction visualization")

def create_error_image(error_message):
    """Create an error image when visualization fails"""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, f"❌ {error_message}", transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='red',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.8))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close()
    buf.seek(0)

    return Image.open(buf)

# -------------------------
# Main Prediction Function
# -------------------------
def predict_emotion_comprehensive(audio):
    """Process audio and predict emotion with comprehensive analysis"""

    if audio is None:
        return (
            None,  # audio playback
            create_error_image("No audio provided"),  # waveform
            create_error_image("No audio provided"),  # mel spectrogram
            create_error_image("No audio provided"),  # mfcc
            create_error_image("No audio provided"),  # prediction
            "❌ No audio provided. Please record or upload audio."  # text result
        )

    try:
        print("🎵 Loading audio file...")
        # Load audio with librosa
        signal, sr = librosa.load(audio, sr=None)
        original_signal = signal.copy()
        original_sr = sr

        print(f"✅ Audio loaded: {len(signal)/sr:.2f}s, {sr}Hz sample rate")

        # Generate waveform plot (before preprocessing)
        print("📊 Generating waveform visualization...")
        waveform_img = create_waveform_plot(original_signal, original_sr, "Original Waveform")

        # Preprocess audio
        print("🔧 Preprocessing audio...")
        processed_signal, sr = preprocess_audio(signal, sr)

        # Generate mel spectrogram
        print("🎼 Generating mel spectrogram...")
        mel_img = create_mel_spectrogram_plot(processed_signal, sr)

        # Generate MFCC visualization
        print("📈 Generating MFCC analysis...")
        mfcc_img = create_mfcc_plot(processed_signal, sr)

        # Extract features
        print("🔍 Extracting audio features...")
        features = extract_comprehensive_features(processed_signal, sr)

        # Reshape for model input (samples, timesteps, features)
        features_reshaped = features.reshape(1, features.shape[0], 1)

        # Make prediction
        print("🧠 Running emotion prediction...")
        prediction = model.predict(features_reshaped, verbose=0)[0]

        # Get the predicted emotion
        predicted_emotion = emotion_labels[np.argmax(prediction)]
        confidence = np.max(prediction) * 100

        # Generate comprehensive prediction visualization
        print("📊 Generating prediction analysis...")
        prediction_img = create_prediction_plot(prediction)

        # Create detailed result text
        result_lines = [
            f"🎭 **Predicted Emotion: {predicted_emotion.upper()}**",
            f"🎯 **Confidence: {confidence:.1f}%**",
            f"⏱️ **Audio Duration: {len(original_signal)/original_sr:.2f} seconds**",
            f"🔊 **Sample Rate: {original_sr} Hz**",
            f"📊 **Features Extracted: {len(features)}**",
            "",
            "**Top 3 Predictions:**"
        ]

        # Add top 3 predictions
        top_3_indices = np.argsort(prediction)[::-1][:3]
        for i, idx in enumerate(top_3_indices, 1):
            emotion = emotion_labels[idx]
            prob = prediction[idx] * 100
            result_lines.append(f"{i}. {emotion.capitalize()}: {prob:.1f}%")

        result_text = "\n".join(result_lines)

        print("✅ Analysis complete!")

        return (
            audio,           # audio playback
            waveform_img,    # waveform plot
            mel_img,         # mel spectrogram
            mfcc_img,        # mfcc plot
            prediction_img,  # prediction analysis
            result_text      # text result
        )

    except Exception as e:
        error_msg = f"❌ Error processing audio: {str(e)}"
        print(error_msg)

        return (
            None,  # audio playback
            create_error_image("Processing failed"),  # waveform
            create_error_image("Processing failed"),  # mel spectrogram
            create_error_image("Processing failed"),  # mfcc
            create_error_image("Processing failed"),  # prediction
            error_msg  # text result
        )

# -------------------------
# Enhanced Gradio Interface
# -------------------------
def create_interface():
    """Create the enhanced Gradio interface"""

    # Custom CSS for better styling
    custom_css = """
    .gradio-container {
        max-width: 1200px !important;
    }

    .main-header {
        text-align: center;
        background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-size: 2.5em;
        font-weight: bold;
        margin-bottom: 0.5em;
    }

    .description {
        text-align: center;
        font-size: 1.1em;
        color: #666;
        margin-bottom: 2em;
        max-width: 800px;
        margin-left: auto;
        margin-right: auto;
    }

    .result-box {
        background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
        border-radius: 10px;
        padding: 20px;
        margin: 10px 0;
    }
    """

    with gr.Blocks(
        title="🎭 Advanced Speech Emotion Recognition",
        theme=gr.themes.Soft(),
        css=custom_css
    ) as interface:

        gr.HTML("""
        <div class="main-header">🎭 Advanced Speech Emotion Recognition</div>
        <div class="description">
            Upload or record audio to analyze emotional content using advanced deep learning techniques.
            This system analyzes multiple audio features including MFCCs, spectral characteristics,
            and temporal dynamics to predict emotions with high accuracy.
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=1):
                # Audio input section
                gr.Markdown("### 🎤 Audio Input")
                audio_input = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Record or Upload Speech (3-10 seconds recommended)",
                    show_download_button=True
                )

                # Analysis button
                analyze_btn = gr.Button(
                    "🚀 Analyze Emotion",
                    variant="primary",
                    size="lg",
                    scale=1
                )

                # Audio playback
                gr.Markdown("### 🔊 Audio Playback")
                audio_output = gr.Audio(
                    label="Processed Audio",
                    show_download_button=True
                )

            with gr.Column(scale=2):
                # Results section
                gr.Markdown("### 📊 Analysis Results")
                result_text = gr.Markdown(
                    "Ready for analysis. Upload or record audio and click 'Analyze Emotion'.",
                    elem_classes=["result-box"]
                )

        # Visualization tabs
        gr.Markdown("### 📈 Detailed Visualizations")
        with gr.Tabs():
            with gr.TabItem("🌊 Audio Waveform"):
                waveform_plot = gr.Image(
                    label="Audio Waveform & Spectrogram",
                    type="pil"
                )

            with gr.TabItem("🎼 Mel Spectrogram"):
                mel_plot = gr.Image(
                    label="Mel Spectrogram Analysis",
                    type="pil"
                )

            with gr.TabItem("📊 MFCC Features"):
                mfcc_plot = gr.Image(
                    label="MFCC Feature Analysis",
                    type="pil"
                )

            with gr.TabItem("🎯 Predictions"):
                prediction_plot = gr.Image(
                    label="Emotion Prediction Analysis",
                    type="pil"
                )

        # Example section
        gr.Markdown("### 🎵 Try These Examples")
        with gr.Row():
            gr.Examples(
                examples=[
                    # Add paths to your example audio files here if you have any
                    # ["path/to/happy_example.wav"],
                    # ["path/to/sad_example.wav"],
                    # ["path/to/angry_example.wav"],
                ],
                inputs=audio_input,
                outputs=[audio_output, waveform_plot, mel_plot, mfcc_plot, prediction_plot, result_text],
                fn=predict_emotion_comprehensive,
                cache_examples=False
            )

        # Information section
        with gr.Accordion("ℹ️ About This System", open=False):
            gr.Markdown("""
            ### How It Works:
            1. **Audio Preprocessing**: Noise reduction, normalization, and trimming
            2. **Feature Extraction**: 62 comprehensive audio features including:
               - MFCC coefficients (20 features)
               - Delta and Delta-Delta MFCCs (40 features)
               - Spectral features (centroid, rolloff, bandwidth)
               - Temporal features (zero-crossing rate, RMS energy)
            3. **Deep Learning Model**: LSTM-based neural network for emotion classification
            4. **Emotion Classes**: Neutral, Calm, Happy, Sad, Angry, Fearful, Disgust, Surprised

            ### Supported Audio Formats:
            - WAV, MP3, FLAC, M4A, and other common formats
            - Recommended: 3-10 seconds of clear speech
            - Sample rate: automatically converted to 22.05 kHz

            ### Tips for Best Results:
            - Use clear, unambiguous emotional speech
            - Minimize background noise
            - Speak naturally and expressively
            - Audio length between 3-10 seconds works best
            """)

        # Connect the button to the function
        analyze_btn.click(
            fn=predict_emotion_comprehensive,
            inputs=[audio_input],
            outputs=[audio_output, waveform_plot, mel_plot, mfcc_plot, prediction_plot, result_text],
            show_progress=True
        )

        # Also trigger on audio input change
        audio_input.change(
            fn=predict_emotion_comprehensive,
            inputs=[audio_input],
            outputs=[audio_output, waveform_plot, mel_plot, mfcc_plot, prediction_plot, result_text],
            show_progress=True
        )

    return interface

# -------------------------
# Launch the application
# -------------------------
if __name__ == "__main__":
    print("🚀 Starting Speech Emotion Recognition System...")
    print(f"📊 Model loaded with {len(emotion_labels)} emotion classes: {list(emotion_labels)}")

    # Create and launch the interface
    interface = create_interface()

    # Launch with sharing enabled for public access
    interface.launch(
        server_name="0.0.0.0",  # Allow external connections
        server_port=7860,       # Standard Gradio port
        share=True,             # Create public link
        debug=True,             # Enable debug mode
        show_error=True         # Show errors in interface
    )

✅ Model loaded successfully!
✅ Emotion classes loaded: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
🚀 Starting Speech Emotion Recognition System...
📊 Model loaded with 8 emotion classes: [np.str_('angry'), np.str_('calm'), np.str_('disgust'), np.str_('fearful'), np.str_('happy'), np.str_('neutral'), np.str_('sad'), np.str_('surprised')]
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://22c000e96a2f4240cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎵 Loading audio file...
✅ Audio loaded: 3.30s, 48000Hz sample rate
📊 Generating waveform visualization...
🔧 Preprocessing audio...
🎼 Generating mel spectrogram...
📈 Generating MFCC analysis...
🔍 Extracting audio features...
✅ Extracted 62 audio features
🧠 Running emotion prediction...
📊 Generating prediction analysis...
✅ Analysis complete!


In [ ]:
import gradio as gr
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import noisereduce as nr
from tensorflow.keras.models import load_model
from PIL import Image
import io
import os
import time
import warnings
warnings.filterwarnings('ignore')

# -------------------------
# Configuration
# -------------------------
SAMPLE_RATE = 22050
MAX_DURATION = 3  # seconds
MAX_LEN = SAMPLE_RATE * MAX_DURATION

# Install required packages (uncomment if running in Colab)
# !pip install gradio librosa matplotlib tensorflow noisereduce

# -------------------------
# Load model and emotion classes
# -------------------------
def load_emotion_model():
    """Load the emotion recognition model with proper error handling"""
    try:
        model = load_model("/content/emotion_recognition_model.h5")
        print("✅ Model loaded successfully!")

        # Try to load the classes from the saved file
        if os.path.exists('emotion_classes.npy'):
            emotion_labels = np.load('emotion_classes.npy', allow_pickle=True)
            print(f"✅ Emotion classes loaded: {emotion_labels}")
        else:
            # Default emotion labels if file not found
            emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
            print("⚠️ Using default emotion labels")

        return model, emotion_labels

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("⚠️ Creating a placeholder model for demonstration")

        # Create a dummy model for demonstration purposes
        from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
        from tensorflow.keras.models import Model

        # More realistic model architecture
        inputs = Input(shape=(62, 1))
        x = LSTM(128, return_sequences=True, dropout=0.2)(inputs)
        x = LSTM(64, dropout=0.2)(x)
        x = Dense(32, activation='relu')(x)
        x = Dropout(0.3)(x)
        outputs = Dense(8, activation='softmax')(x)

        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

        # Default emotion labels
        emotion_labels = np.array(['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised'])
        print("⚠️ Using placeholder model and default emotion labels")

        return model, emotion_labels

# Load model and labels globally
model, emotion_labels = load_emotion_model()

# -------------------------
# Audio Processing Functions
# -------------------------
def preprocess_audio(signal, sr):
    """Apply comprehensive preprocessing to the audio signal"""
    try:
        # Ensure the sample rate is correct
        if sr != SAMPLE_RATE:
            signal = librosa.resample(signal, orig_sr=sr, target_sr=SAMPLE_RATE)
            sr = SAMPLE_RATE

        # Remove silence from beginning and end
        signal, _ = librosa.effects.trim(signal, top_db=20)

        # Normalize audio to [-1, 1] range
        signal = librosa.util.normalize(signal)

        # Apply noise reduction
        signal = nr.reduce_noise(y=signal, sr=sr, stationary=False)

        # Apply pre-emphasis filter to balance the frequency spectrum
        signal = np.append(signal[0], signal[1:] - 0.97 * signal[:-1])

        return signal, sr
    except Exception as e:
        print(f"Warning in preprocessing: {e}")
        # Return original signal if preprocessing fails
        return librosa.util.normalize(signal), sr

def extract_comprehensive_features(signal, sr):
    """Extract comprehensive audio features for emotion recognition"""
    try:
        # Handle audio length - pad or truncate to fixed length
        if len(signal) > MAX_LEN:
            signal = signal[:MAX_LEN]
        else:
            signal = np.pad(signal, (0, MAX_LEN - len(signal)))

        features = []

        # 1. MFCCs (Mel-frequency cepstral coefficients) - 20 features
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
        mfccs_mean = np.mean(mfccs.T, axis=0)
        mfccs_std = np.std(mfccs.T, axis=0)
        features.extend(mfccs_mean)
        features.extend(mfccs_std)  # Add standard deviation for more robustness

        # 2. Delta MFCCs (1st order derivatives) - 20 features
        delta_mfccs = librosa.feature.delta(mfccs)
        delta_mfccs_mean = np.mean(delta_mfccs.T, axis=0)
        features.extend(delta_mfccs_mean)

        # 3. Delta-Delta MFCCs (2nd order derivatives) - 20 features
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        delta2_mfccs_mean = np.mean(delta2_mfccs.T, axis=0)
        features.extend(delta2_mfccs_mean)

        # 4. Zero-crossing rate - voice characteristics
        zcr = librosa.feature.zero_crossing_rate(y=signal, frame_length=2048, hop_length=512)
        features.append(np.mean(zcr))
        features.append(np.std(zcr))

        # 5. Root mean square energy - volume/intensity
        rmse = librosa.feature.rms(y=signal, frame_length=2048, hop_length=512)
        features.append(np.mean(rmse))
        features.append(np.std(rmse))

        # 6. Spectral centroid - brightness of sound
        spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_centroid))
        features.append(np.std(spectral_centroid))

        # 7. Spectral rolloff - frequency below which 85% of energy is contained
        spectral_rolloff = librosa.feature.spectral_rolloff(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_rolloff))
        features.append(np.std(spectral_rolloff))

        # 8. Spectral bandwidth - width of the frequency band
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=sr, hop_length=512)
        features.append(np.mean(spectral_bandwidth))
        features.append(np.std(spectral_bandwidth))

        # Convert to numpy array and ensure we have exactly 62 features
        features = np.array(features)

        # Pad or truncate to exactly 62 features to match model input
        if len(features) < 62:
            features = np.pad(features, (0, 62 - len(features)))
        elif len(features) > 62:
            features = features[:62]

        print(f"✅ Extracted {len(features)} audio features")
        return features

    except Exception as e:
        print(f"❌ Error in feature extraction: {e}")
        # Return dummy features if extraction fails
        return np.zeros(62)  # Match expected feature count

# -------------------------
# Enhanced Visualization Functions
# -------------------------
def create_waveform_plot(signal, sr, title="Audio Waveform"):
    """Generate enhanced waveform plot and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

        # Time axis
        time_axis = np.linspace(0, len(signal) / sr, len(signal))

        # Original waveform
        ax1.plot(time_axis, signal, color='#00d4ff', linewidth=0.8, alpha=0.8)
        ax1.fill_between(time_axis, signal, alpha=0.3, color='#00d4ff')
        ax1.set_title(f'{title} - Time Domain', fontsize=12, color='white', pad=10)
        ax1.set_xlabel('Time (seconds)', color='white')
        ax1.set_ylabel('Amplitude', color='white')
        ax1.grid(True, alpha=0.3)
        ax1.tick_params(colors='white')

        # Spectrogram
        D = librosa.amplitude_to_db(np.abs(librosa.stft(signal)), ref=np.max)
        img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz',
                                      ax=ax2, cmap='plasma', alpha=0.8)
        ax2.set_title('Spectrogram - Frequency Domain', fontsize=12, color='white', pad=10)
        ax2.tick_params(colors='white')
        plt.colorbar(img, ax=ax2, format='%+2.0f dB')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating waveform plot: {e}")
        return create_error_image("Error generating waveform visualization")

def create_mel_spectrogram_plot(signal, sr):
    """Generate enhanced Mel Spectrogram visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # Mel Spectrogram
        S = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128,
                                         n_fft=2048, hop_length=512, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)

        img1 = librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel',
                                       fmax=8000, ax=ax1, cmap='viridis')
        ax1.set_title('Mel Spectrogram', fontsize=14, color='white', pad=15)
        ax1.tick_params(colors='white')
        plt.colorbar(img1, ax=ax1, format='%+2.0f dB')

        # Mel filter bank visualization
        mel_filters = librosa.filters.mel(sr=sr, n_fft=2048, n_mels=10, fmax=8000)
        ax2.plot(mel_filters.T, alpha=0.8, linewidth=1.5)
        ax2.set_title('Mel Filter Bank (First 10 filters)', fontsize=14, color='white', pad=15)
        ax2.set_xlabel('Frequency Bins', color='white')
        ax2.set_ylabel('Filter Response', color='white')
        ax2.grid(True, alpha=0.3)
        ax2.tick_params(colors='white')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating mel spectrogram: {e}")
        return create_error_image("Error generating mel spectrogram")

def create_mfcc_plot(signal, sr):
    """Generate enhanced MFCC visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

        # MFCCs
        mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
        img1 = librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax1, cmap='coolwarm')
        ax1.set_title('MFCCs (Mel-frequency Cepstral Coefficients)', fontsize=12, color='white')
        ax1.set_ylabel('MFCC Coefficients', color='white')
        ax1.tick_params(colors='white')
        plt.colorbar(img1, ax=ax1)

        # Delta MFCCs
        delta_mfccs = librosa.feature.delta(mfccs)
        img2 = librosa.display.specshow(delta_mfccs, sr=sr, x_axis='time', ax=ax2, cmap='coolwarm')
        ax2.set_title('Delta MFCCs (1st derivatives)', fontsize=12, color='white')
        ax2.set_ylabel('Delta MFCC', color='white')
        ax2.tick_params(colors='white')
        plt.colorbar(img2, ax=ax2)

        # Delta-Delta MFCCs
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        img3 = librosa.display.specshow(delta2_mfccs, sr=sr, x_axis='time', ax=ax3, cmap='coolwarm')
        ax3.set_title('Delta-Delta MFCCs (2nd derivatives)', fontsize=12, color='white')
        ax3.set_ylabel('Delta-Delta MFCC', color='white')
        ax3.set_xlabel('Time (s)', color='white')
        ax3.tick_params(colors='white')
        plt.colorbar(img3, ax=ax3)

        # MFCC statistics
        mfcc_mean = np.mean(mfccs.T, axis=0)
        mfcc_std = np.std(mfccs.T, axis=0)

        x_pos = np.arange(len(mfcc_mean))
        ax4.bar(x_pos - 0.2, mfcc_mean, 0.4, label='Mean', alpha=0.8, color='#ff6b6b')
        ax4.bar(x_pos + 0.2, mfcc_std, 0.4, label='Std Dev', alpha=0.8, color='#4ecdc4')
        ax4.set_title('MFCC Statistics', fontsize=12, color='white')
        ax4.set_xlabel('MFCC Coefficient Index', color='white')
        ax4.set_ylabel('Value', color='white')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        ax4.tick_params(colors='white')

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating MFCC plot: {e}")
        return create_error_image("Error generating MFCC visualization")

def create_prediction_plot(prediction_probs, feature_importance=None):
    """Generate comprehensive prediction visualization and return PIL Image"""
    try:
        plt.style.use('dark_background')
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

        # 1. Horizontal bar chart of predictions
        sorted_indices = np.argsort(prediction_probs)
        sorted_probs = prediction_probs[sorted_indices]
        sorted_labels = emotion_labels[sorted_indices]

        colors = plt.cm.viridis(np.linspace(0, 1, len(sorted_probs)))
        bars = ax1.barh(sorted_labels, sorted_probs * 100, color=colors, alpha=0.8)
        ax1.set_xlabel('Probability (%)', color='white', fontsize=12)
        ax1.set_title('Emotion Prediction Probabilities', color='white', fontsize=14, pad=15)
        ax1.tick_params(colors='white')
        ax1.grid(True, alpha=0.3, axis='x')

        # Add percentage labels on bars
        for bar, prob in zip(bars, sorted_probs):
            width = bar.get_width()
            ax1.text(width + 1, bar.get_y() + bar.get_height()/2,
                    f'{prob*100:.1f}%', ha='left', va='center', color='white', fontsize=10)

        # 2. Pie chart for top emotions
        top_n = 5
        top_indices = np.argsort(prediction_probs)[::-1][:top_n]
        top_probs = prediction_probs[top_indices]
        top_labels = emotion_labels[top_indices]

        # Add "others" category if needed
        others_prob = 1 - np.sum(top_probs)
        if others_prob > 0.01:  # Only add if significant
            top_probs = np.append(top_probs, others_prob)
            top_labels = np.append(top_labels, 'Others')

        colors_pie = plt.cm.Set3(np.linspace(0, 1, len(top_probs)))
        wedges, texts, autotexts = ax2.pie(top_probs, labels=top_labels, autopct='%1.1f%%',
                                          startangle=90, colors=colors_pie,
                                          textprops={'color': 'white', 'fontsize': 10})
        ax2.set_title('Top Emotion Predictions', color='white', fontsize=14, pad=15)

        # 3. Confidence analysis
        max_prob = np.max(prediction_probs)
        entropy = -np.sum(prediction_probs * np.log2(prediction_probs + 1e-10))
        max_entropy = np.log2(len(emotion_labels))
        normalized_entropy = entropy / max_entropy

        confidence_metrics = ['Max Probability', 'Certainty\n(1 - Entropy)', 'Top-2 Gap']
        top2_gap = sorted_probs[-1] - sorted_probs[-2] if len(sorted_probs) > 1 else max_prob
        confidence_values = [max_prob, 1 - normalized_entropy, top2_gap]

        bars3 = ax3.bar(confidence_metrics, confidence_values,
                       color=['#ff6b6b', '#4ecdc4', '#45b7d1'], alpha=0.8)
        ax3.set_title('Prediction Confidence Metrics', color='white', fontsize=14, pad=15)
        ax3.set_ylabel('Score', color='white')
        ax3.tick_params(colors='white')
        ax3.grid(True, alpha=0.3, axis='y')

        # Add value labels on bars
        for bar, value in zip(bars3, confidence_values):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{value:.3f}', ha='center', va='bottom', color='white', fontsize=11)

        # 4. Prediction distribution
        ax4.plot(emotion_labels, prediction_probs * 100, 'o-',
                color='#ffa726', linewidth=2, markersize=8, alpha=0.8)
        ax4.fill_between(range(len(emotion_labels)), prediction_probs * 100,
                        alpha=0.3, color='#ffa726')
        ax4.set_title('Prediction Distribution', color='white', fontsize=14, pad=15)
        ax4.set_ylabel('Probability (%)', color='white')
        ax4.set_xlabel('Emotions', color='white')
        ax4.tick_params(colors='white', axis='x', rotation=45)
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        fig.patch.set_facecolor('#1e1e1e')

        # Convert to PIL Image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                   facecolor='#1e1e1e', edgecolor='none')
        plt.close()
        buf.seek(0)

        return Image.open(buf)

    except Exception as e:
        print(f"Error creating prediction plot: {e}")
        return create_error_image("Error generating prediction visualization")

def create_error_image(error_message):
    """Create an error image when visualization fails"""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, f"❌ {error_message}", transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='red',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.8))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close()
    buf.seek(0)

    return Image.open(buf)

# -------------------------
# Main Prediction Function
# -------------------------
def predict_emotion_comprehensive(audio):
    """Process audio and predict emotion with comprehensive analysis"""

    if audio is None:
        return (
            None,  # audio playback
            create_error_image("No audio provided"),  # waveform
            create_error_image("No audio provided"),  # mel spectrogram
            create_error_image("No audio provided"),  # mfcc
            create_error_image("No audio provided"),  # prediction
            "❌ No audio provided. Please record or upload audio."  # text result
        )

    try:
        print("🎵 Loading audio file...")
        # Load audio with librosa
        signal, sr = librosa.load(audio, sr=None)
        original_signal = signal.copy()
        original_sr = sr

        print(f"✅ Audio loaded: {len(signal)/sr:.2f}s, {sr}Hz sample rate")

        # Generate waveform plot (before preprocessing)
        print("📊 Generating waveform visualization...")
        waveform_img = create_waveform_plot(original_signal, original_sr, "Original Waveform")

        # Preprocess audio
        print("🔧 Preprocessing audio...")
        processed_signal, sr = preprocess_audio(signal, sr)

        # Generate mel spectrogram
        print("🎼 Generating mel spectrogram...")
        mel_img = create_mel_spectrogram_plot(processed_signal, sr)

        # Generate MFCC visualization
        print("📈 Generating MFCC analysis...")
        mfcc_img = create_mfcc_plot(processed_signal, sr)

        # Extract features
        print("🔍 Extracting audio features...")
        features = extract_comprehensive_features(processed_signal, sr)

        # Reshape for model input (samples, timesteps, features)
        features_reshaped = features.reshape(1, features.shape[0], 1)

        # Make prediction
        print("🧠 Running emotion prediction...")
        prediction = model.predict(features_reshaped, verbose=0)[0]

        # Get the predicted emotion
        predicted_emotion = emotion_labels[np.argmax(prediction)]
        confidence = np.max(prediction) * 100

        # Generate comprehensive prediction visualization
        print("📊 Generating prediction analysis...")
        prediction_img = create_prediction_plot(prediction)

        # Create detailed result text
        result_lines = [
            f"🎭 **Predicted Emotion: {predicted_emotion.upper()}**",
            f"🎯 **Confidence: {confidence:.1f}%**",
            f"⏱️ **Audio Duration: {len(original_signal)/original_sr:.2f} seconds**",
            f"🔊 **Sample Rate: {original_sr} Hz**",
            f"📊 **Features Extracted: {len(features)}**",
            "",
            "**Top 3 Predictions:**"
        ]

        # Add top 3 predictions
        top_3_indices = np.argsort(prediction)[::-1][:3]
        for i, idx in enumerate(top_3_indices, 1):
            emotion = emotion_labels[idx]
            prob = prediction[idx] * 100
            result_lines.append(f"{i}. {emotion.capitalize()}: {prob:.1f}%")

        result_text = "\n".join(result_lines)

        print("✅ Analysis complete!")

        return (
            audio,           # audio playback
            waveform_img,    # waveform plot
            mel_img,         # mel spectrogram
            mfcc_img,        # mfcc plot
            prediction_img,  # prediction analysis
            result_text      # text result
        )

    except Exception as e:
        error_msg = f"❌ Error processing audio: {str(e)}"
        print(error_msg)

        return (
            None,  # audio playback
            create_error_image("Processing failed"),  # waveform
            create_error_image("Processing failed"),  # mel spectrogram
            create_error_image("Processing failed"),  # mfcc
            create_error_image("Processing failed"),  # prediction
            error_msg  # text result
        )

# -------------------------
# Enhanced Gradio Interface
# -------------------------
def create_interface():
    """Create the enhanced Gradio interface"""

    # Custom CSS for better styling
    custom_css = """
    .gradio-container {
        max-width: 1200px !important;
    }

    .main-header {
        text-align: center;
        background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-size: 2.5em;
        font-weight: bold;
        margin-bottom: 0.5em;
    }

    .description {
        text-align: center;
        font-size: 1.1em;
        color: #666;
        margin-bottom: 2em;
        max-width: 800px;
        margin-left: auto;
        margin-right: auto;
    }

    .result-box {
        background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
        border-radius: 10px;
        padding: 20px;
        margin: 10px 0;
    }
    """

    with gr.Blocks(
        title="🎭 Advanced Speech Emotion Recognition",
        theme=gr.themes.Soft(),
        css=custom_css
    ) as interface:

        gr.HTML("""
        <div class="main-header">🎭 Advanced Speech Emotion Recognition</div>
        <div class="description">
            Upload or record audio to analyze emotional content using advanced deep learning techniques.
            This system analyzes multiple audio features including MFCCs, spectral characteristics,
            and temporal dynamics to predict emotions with high accuracy.
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=1):
                # Audio input section
                gr.Markdown("### 🎤 Audio Input")
                audio_input = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Record or Upload Speech (3-10 seconds recommended)",
                    show_download_button=True
                )

                # Analysis button
                analyze_btn = gr.Button(
                    "🚀 Analyze Emotion",
                    variant="primary",
                    size="lg",
                    scale=1
                )

                # Audio playback
                gr.Markdown("### 🔊 Audio Playback")
                audio_output = gr.Audio(
                    label="Processed Audio",
                    show_download_button=True
                )

            with gr.Column(scale=2):
                # Results section
                gr.Markdown("### 📊 Analysis Results")
                result_text = gr.Markdown(
                    "Ready for analysis. Upload or record audio and click 'Analyze Emotion'.",
                    elem_classes=["result-box"]
                )

        # Visualization tabs
        gr.Markdown("### 📈 Detailed Visualizations")
        with gr.Tabs():
            with gr.TabItem("🌊 Audio Waveform"):
                waveform_plot = gr.Image(
                    label="Audio Waveform & Spectrogram",
                    type="pil"
                )

            with gr.TabItem("🎼 Mel Spectrogram"):
                mel_plot = gr.Image(
                    label="Mel Spectrogram Analysis",
                    type="pil"
                )

            with gr.TabItem("📊 MFCC Features"):
                mfcc_plot = gr.Image(
                    label="MFCC Feature Analysis",
                    type="pil"
                )

            with gr.TabItem("🎯 Predictions"):
                prediction_plot = gr.Image(
                    label="Emotion Prediction Analysis",
                    type="pil"
                )

        # Example section
        gr.Markdown("### 🎵 Try These Examples")